# model-v3-final
Generated from `code/business_entity_resolution/src/run_final.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0 lightgbm==4.7.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/train_model.py
"""Train the final matching model (two-stage LightGBM [+ CatBoost], see pipeline.py) and check it on validation.

  python src/train_model.py --input DIR --config best_config.json --out-dir artifacts/model_final [--final]

--input is searched for: g25_split.parquet, train_source1.parquet and train_ground_truth.parquet (organiser data),
train_candidates.parquet (blocking v3), features_train.parquet / features_val.parquet (features-v2) and
normalised/{train,test}_source{1,2,3}.parquet (norm-v3). --config is the chosen setup from run_model_v3.py
(stage-2 features, LightGBM parameters, seeds, blend, selection).
Training S1: the train split without the blocking pruner's 100k S1 (their pruner-derived inputs are in-sample);
with --final also the validation tune half (even S1 id). The validation report half (odd id) is never trained on
and is the final check: macro F0.5 overall, per country, singletons / non-singletons, precision, recall.
Writes stage1_fold*.txt, stage2_lgb_seed*.txt, [catboost.cbm], model_config.json, train_report.md.
"""
import argparse
import json
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:
    sys.path.insert(0, "/tmp/src")
from model import feature_sets  # noqa: E402
from pipeline import blend, counts, find, normalised_files, raw_file, records, stage2_frame  # noqa: E402
from selection import f05_table, select, summary  # noqa: E402
from two_stage import PARAMS, fit, fold_of, predict, stage1_oof, stage1_predict  # noqa: E402

SEED = 42
BLEND = {"CatBoost": "cb", "blend: mean prob": "mean", "blend: rank mean": "rank"}
CB_PARAMS = dict(iterations=3000, learning_rate=0.08, depth=8, loss_function="Logloss", od_type="Iter", od_wait=100,
                 thread_count=-1, verbose=0)
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:7.1f}s peak {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:5.1f} GB] {msg}", flush=True)


def md(df: pl.DataFrame) -> str:
    f = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(f(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--config", required=True, type=Path, help="best_config.json from run_model_v3.py (a folder is searched)")
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--final", action="store_true", help="also train on the validation tune half")
    ap.add_argument("--pruner-s1", type=int, default=100_000)
    a = ap.parse_args()
    out = a.out_dir
    out.mkdir(parents=True, exist_ok=True)
    cfg = json.loads((find(a.config, "best_config.json") if a.config.is_dir() else a.config).read_text())
    how = BLEND.get(cfg["setup"], "lgb")
    rd = lambda name, cols=None: pl.read_parquet(raw_file(a.input, name), columns=cols)

    split = rd("g25_split.parquet")
    trn = split.filter(pl.col("role") == "train")["source1_entity_id"]
    pruner_s1 = trn.sample(min(a.pruner_s1, trn.len()), seed=SEED)
    val_s1 = split.filter(pl.col("role") == "val").select(pl.col("source1_entity_id").alias("s1")).join(
        rd("train_source1.parquet", ["entity_id", "country"]).rename({"entity_id": "s1"}), on="s1", how="left")
    gt = rd("train_ground_truth.parquet")
    truth = (gt.join(val_s1, left_on="source1_entity_id", right_on="s1", how="semi")
             .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
             .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != "")))
    half = pl.col("s1").str.extract(r"(\d+)$").cast(pl.Int64) % 2
    s1_tune, s1_rep = val_s1.filter(half == 0), val_s1.filter(half == 1)

    ftr = rd("features_train.parquet")
    c_feats = feature_sets(ftr.columns)["C"]
    ftr = ftr.join(rd("train_candidates.parquet", ["s1", "cand", "is_match"]), on=["s1", "cand"], how="left")
    ftr = ftr.filter(~pl.col("s1").is_in(pruner_s1.implode()))
    fva = rd("features_val.parquet").join(truth.with_columns(pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left") \
        .with_columns(pl.col("is_match").fill_null(False))
    if a.final:
        ftr = pl.concat([ftr, fva.join(s1_tune, on="s1", how="semi").select(ftr.columns)])
    ftr = ftr.sort("s1", "cand")
    rep = fva.join(s1_rep, on="s1", how="semi").sort("s1", "cand")
    y = ftr["is_match"].cast(pl.Int8).to_numpy()
    log(f"training {ftr.height:,} pairs / {ftr['s1'].n_unique():,} S1 (final={a.final}); report half {rep.height:,} pairs")

    # stage 1
    oof, models1 = stage1_oof(ftr.select("s1", *c_feats), y, c_feats, log=log)
    for i, b in enumerate(models1):
        b.save_model(str(out / f"stage1_fold{i}.txt"), num_iteration=b.best_iteration)
    # stage-2 features
    norm = normalised_files(a.input)
    cnt = counts(norm)
    s1_rec, c_rec = records(norm, "train", pl.concat([ftr["s1"], rep["s1"]]).unique(), pl.concat([ftr["cand"], rep["cand"]]).unique())
    X = stage2_frame(ftr, oof, s1_rec, c_rec, cnt, c_feats)
    Xr = stage2_frame(rep, stage1_predict(models1, rep, c_feats), s1_rec, c_rec, cnt, c_feats)
    fs = cfg["features"]
    m, mr = X.select(fs).to_numpy(), Xr.select(fs).to_numpy()
    es = fold_of(ftr["s1"], 10, salt=7) == 0
    params = {k: v for k, v in cfg["lgb_params"].items() if k not in PARAMS or PARAMS[k] != v}
    pl_, pr_cb = [], None
    for s in range(int(cfg["seeds"])):
        b = fit(m, y, es, fs, seed=SEED + 100 + s, params=params)
        b.save_model(str(out / f"stage2_lgb_seed{s}.txt"), num_iteration=b.best_iteration)
        pl_.append(predict(b, mr))
        log(f"stage-2 LightGBM seed {s}: best iteration {b.best_iteration}")
    if how != "lgb":
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier(**CB_PARAMS, random_seed=SEED)
        cb.fit(m[~es], y[~es], eval_set=(m[es], y[es]))
        cb.save_model(str(out / "catboost.cbm"))
        pr_cb = cb.predict_proba(mr)[:, 1]
        log(f"CatBoost best iteration {cb.get_best_iteration()}")
    p = blend(np.mean(pl_, axis=0), pr_cb, how)

    # final check on the report half (fixed selection from the experiments)
    sel_cfg = cfg["selection"]
    sv = rep.select("s1", "cand").with_columns(pl.Series("p", p))
    sm = summary(f05_table(select(sv, **sel_cfg), truth, s1_rep), by="country")
    log("report half:\n" + str(sm))
    sm.write_csv(out / "report_half.csv")
    (out / "model_config.json").write_text(json.dumps({
        "stage1_features": c_feats, "stage2_features": fs, "lgb_params": cfg["lgb_params"], "seeds": int(cfg["seeds"]),
        "blend": how, "selection": sel_cfg, "final": a.final, "setup": cfg["setup"],
        "report_half": {r["group"]: {k: r[k] for k in ("macro_f05", "precision", "recall", "pred_per_s1")} for r in sm.iter_rows(named=True)},
        "trained_s1": ftr["s1"].n_unique()}, indent=1, default=str))
    (out / "train_report.md").write_text(f"# Final model ({cfg['setup']}, final={a.final})\n\n{md(sm)}\n")
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/src/predict.py
"""Test inference with the model trained by train_model.py; writes both official output files.

  python src/predict.py --input DIR --model-dir artifacts/model_final --out-dir ../../output \
      [--validator ../../utils/validate_submission.py] [--fallback ../../output/fallback_v2/matching_results.tsv]

Per country (candidate lists never cross countries): stage 1 (fold-model average) -> set features -> B features
-> stage 2 (LightGBM seeds [+ CatBoost]) -> blend. Then the selection from training (threshold, margin) with ONE
OWNER per S2/S3 over all test S1. candidate_pairs.tsv = the blocking v3 test candidates, exactly the pairs scored;
matches are a subset of it. Also writes predict_report.md: matches per S1 by country, share without a match,
10 random French S1 with their matches, and the change against a previous matching file (--fallback).
"""
import argparse
import json
import resource
import sys
import tempfile
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:
    sys.path.insert(0, "/tmp/src")
from pipeline import blend, counts, normalised_files, raw_file, records, stage2_frame  # noqa: E402
from selection import select  # noqa: E402
from submission import write_submission  # noqa: E402

SEED = 42
T0 = time.time()


def log(msg: str) -> None:
    print(f"[{time.time() - T0:7.1f}s peak {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:5.1f} GB] {msg}", flush=True)


def md(df: pl.DataFrame, n: int = 80) -> str:
    df = df.head(n)
    f = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(f(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def ids_sets(path: Path) -> pl.DataFrame:
    return (pl.read_csv(path, separator="\t", quote_char=None, infer_schema=False)
            .select(pl.col("source1_entity_id").alias("s1"),
                    pl.col("matched_entity_ids").fill_null("").str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.sort().alias("m")))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, type=Path)
    ap.add_argument("--model-dir", required=True, type=Path)
    ap.add_argument("--out-dir", required=True, type=Path)
    ap.add_argument("--validator", default=None)
    ap.add_argument("--fallback", default=None, type=Path)
    a = ap.parse_args()
    a.out_dir.mkdir(parents=True, exist_ok=True)
    md_dir = a.model_dir if (a.model_dir / "model_config.json").exists() else next(a.model_dir.rglob("model_config.json")).parent
    cfg = json.loads((md_dir / "model_config.json").read_text())
    s1m = [lgb.Booster(model_file=str(p)) for p in sorted(md_dir.glob("stage1_fold*.txt"))]
    s2m = [lgb.Booster(model_file=str(p)) for p in sorted(md_dir.glob("stage2_lgb_seed*.txt"))]
    cb = None
    if cfg["blend"] != "lgb":
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier()
        cb.load_model(str(md_dir / "catboost.cbm"))
    c_feats, fs = cfg["stage1_features"], cfg["stage2_features"]
    log(f"model: {len(s1m)} stage-1 folds, {len(s2m)} stage-2 seeds, blend {cfg['blend']}, selection {cfg['selection']}")

    s1_all = pl.read_parquet(raw_file(a.input, "test_source1.parquet"), columns=["entity_id", "country"]).rename({"entity_id": "s1"})
    norm = normalised_files(a.input)
    cnt = counts(norm)
    fpath = raw_file(a.input, "features_test.parquet")
    parts = []
    for country in sorted(s1_all["country"].unique().to_list()):
        fx = pl.scan_parquet(fpath).join(s1_all.filter(pl.col("country") == country).lazy().select("s1"), on="s1", how="semi").collect().sort("s1", "cand")
        if not fx.height:
            continue
        p1 = np.mean([b.predict(fx.select(c_feats).to_numpy()) for b in s1m], axis=0)
        s1_rec, c_rec = records(norm, "test", fx["s1"].unique(), fx["cand"].unique())
        X = stage2_frame(fx, p1, s1_rec, c_rec, cnt, c_feats)
        m = X.select(fs).to_numpy()
        p_lgb = np.mean([b.predict(m) for b in s2m], axis=0)
        p_cb = cb.predict_proba(m)[:, 1] if cb is not None else None
        parts.append(fx.select("s1", "cand").with_columns(pl.Series("p_lgb", p_lgb), pl.Series("p_cb", p_cb if p_cb is not None else p_lgb)))
        log(f"{country}: {fx.height:,} pairs scored")
        del fx, X, m, s1_rec, c_rec
    st = pl.concat(parts)
    st = st.with_columns(pl.Series("p", blend(st["p_lgb"].to_numpy(), st["p_cb"].to_numpy() if cb is not None else None, cfg["blend"])))
    sel = {**cfg["selection"], "one_owner": True}  # an S2/S3 matches at most one S1; on test all S1 compete
    matched = select(st.select("s1", "cand", "p"), **sel)
    log(f"{matched.height:,} matches")
    st.select("s1", "cand", "p").write_parquet(a.out_dir.parent / "test_scores.parquet")  # pair probabilities (for blends)

    cand = pl.read_parquet(raw_file(a.input, "test_candidates.parquet"), columns=["s1", "cand", "p_u50"])
    assert cand.select("s1", "cand").sort("s1", "cand").equals(st.select("s1", "cand").sort("s1", "cand")), "scored pairs != candidate set"
    lists = cand.sort(["s1", "p_u50", "cand"], descending=[False, True, False], nulls_last=True).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    ml = matched.sort(["s1", "p"], descending=[False, True]).group_by("s1", maintain_order=True).agg(pl.col("cand"))
    write_submission(a.out_dir, s1_all["s1"].to_list(), dict(zip(ml["s1"].to_list(), ml["cand"].to_list())),
                     dict(zip(lists["s1"].to_list(), lists["cand"].to_list())))
    log(f"wrote {a.out_dir}/matching_results.tsv and candidate_pairs.tsv")

    report = [f"# Test predictions ({cfg['setup']})\n", f"Matches: {matched.height:,} for {s1_all.height:,} S1; selection {sel}.\n"]
    if a.validator:
        import importlib.util
        spec = importlib.util.spec_from_file_location("vs", a.validator); vs = importlib.util.module_from_spec(spec); spec.loader.exec_module(vs)
        with tempfile.TemporaryDirectory() as td:
            for k in (1, 2, 3):
                pl.read_parquet(raw_file(a.input, f"test_source{k}.parquet"), columns=["entity_id"]).write_csv(Path(td) / f"test_source{k}.tsv", separator="\t")
            e, w = vs.validate(str(a.out_dir / "matching_results.tsv"), str(a.out_dir / "candidate_pairs.tsv"), td, check_ids=True)
        report.append(f"Validator (--check-ids): {len(e)} errors, {len(w)} warnings {e[:3]}\n")
        log(f"validator: {len(e)} errors, {len(w)} warnings")
    n = s1_all.join(matched.group_by("s1").len("n"), on="s1", how="left").with_columns(pl.col("n").fill_null(0))
    pc = n.group_by("country").agg(pl.len().alias("s1"), pl.col("n").mean().alias("matches_per_s1"),
                                   (pl.col("n") == 0).mean().alias("share_no_match")).sort("country")
    report += ["## Matches per S1\n", md(pc) + "\n"]
    fr = s1_all.filter(pl.col("country") == "France").sample(min(10, int((s1_all["country"] == "France").sum())), seed=SEED)
    if fr.height:
        names1 = pl.read_parquet(raw_file(a.input, "test_source1.parquet"), columns=["entity_id", "business_name", "business_address"]).rename({"entity_id": "s1"})
        ex = matched.join(fr, on="s1", how="semi")
        cn = pl.concat([pl.read_parquet(raw_file(a.input, f"test_source{k}.parquet"), columns=["entity_id", "business_name", "business_address"])
                        .join(ex.select(pl.col("cand").alias("entity_id")), on="entity_id", how="semi") for k in (2, 3)])
        ex = (fr.join(names1, on="s1").join(ex, on="s1", how="left")
              .join(cn.rename({"entity_id": "cand", "business_name": "c_name", "business_address": "c_addr"}), on="cand", how="left")
              .select("s1", "business_name", "business_address", "p", "c_name", "c_addr"))
        report += ["## 10 random French S1\n", md(ex) + "\n"]
    if a.fallback and a.fallback.exists():
        x = ids_sets(a.out_dir / "matching_results.tsv").join(ids_sets(a.fallback), on="s1", suffix="_fb").join(s1_all, on="s1")
        ch = x.with_columns((pl.col("m") != pl.col("m_fb")).alias("changed")).group_by("country").agg(
            pl.len().alias("s1"), pl.col("changed").sum().alias("s1_changed"), pl.col("changed").mean().alias("share")).sort("country")
        report += [f"## Change vs {a.fallback.name}\n", md(ch) + "\n"]
    (a.out_dir / "predict_report.md").write_text("\n".join(report))
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/src/pipeline.py
"""Shared pieces of the final matching model (train_model.py, predict.py).

Design (model v3):
- stage 1: LightGBM on feature set C (features-v2 without the method count and pruner-derived features), 5 folds
  grouped by S1; out-of-fold probabilities on training pairs, the fold-model average elsewhere;
- stage 2 features: C + set features (two_stage.py: the pair's stage-1 probability relative to the other candidates
  of its S1 list) + targeted B features (pair_extra.py);
- stage 2: LightGBM (several seeds averaged) and optionally CatBoost, blended as configured;
- selection (selection.py): threshold, margin to the S1's best, one owner per S2/S3 (always on test).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import polars as pl

from features import prepare_records
from pair_extra import country_counts, extra_features
from two_stage import SET_FEATURES, set_features

REC_COLS = ["entity_id", "country", "full_name", "core_name", "legal", "addr_norm", "city", "state", "numbers"]


def find(root: Path, name: str, parent: str | None = None, not_parent: tuple = ()) -> Path:
    hits = sorted(p for p in root.rglob(name) if (parent is None or p.parent.name == parent) and p.parent.name not in not_parent)
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


def raw_file(root: Path, name: str) -> Path:
    """The organiser / blocking / features copy of a file (never the normalised or experiment copies)."""
    return find(root, name, not_parent=("normalised", "features_extra", "aug"))


def normalised_files(root: Path) -> dict[tuple[str, int], Path]:
    return {(sd, k): find(root, f"{sd}_source{k}.parquet", parent="normalised") for sd in ("train", "test") for k in (1, 2, 3)}


def counts(norm: dict) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """IDF, name-per-state and address counts over ALL train + test records (no labels)."""
    return country_counts(pl.concat([pl.scan_parquet(p).select("country", "state", "core_name", "addr_norm") for p in norm.values()]))


def records(norm: dict, side: str, s1_ids: pl.Series, cand_ids: pl.Series) -> tuple[pl.DataFrame, pl.DataFrame]:
    s1 = prepare_records(pl.scan_parquet(norm[(side, 1)]).select(REC_COLS).filter(pl.col("entity_id").is_in(s1_ids.implode())).collect())
    c = prepare_records(pl.concat([pl.scan_parquet(norm[(side, k)]).select(REC_COLS).filter(pl.col("entity_id").is_in(cand_ids.implode())).collect()
                                   for k in (2, 3)]))
    return s1, c


def stage2_frame(fx: pl.DataFrame, p1: np.ndarray, s1_rec: pl.DataFrame, c_rec: pl.DataFrame, cnt, stage1_feats: list[str]) -> pl.DataFrame:
    """fx: s1, cand + features-v2 columns (whole S1 lists, sorted by s1, cand). Returns s1, cand + C + set + B."""
    sf = set_features(fx, p1, c_rec.select("entity_id", "full_name", "addr_norm", "numbers"))
    bx = extra_features(fx.select("s1", "cand"), s1_rec, c_rec, *cnt)
    return fx.select("s1", "cand", *stage1_feats).hstack(sf.select(SET_FEATURES)).hstack(bx.drop("s1", "cand"))


def blend(p_lgb: np.ndarray, p_cb: np.ndarray | None, how: str) -> np.ndarray:
    if p_cb is None or how == "lgb":
        return p_lgb
    if how == "cb":
        return p_cb
    if how == "mean":
        return (p_lgb + p_cb) / 2
    if how == "rank":
        n = len(p_lgb)
        return (pl.Series(p_lgb).rank().to_numpy() + pl.Series(p_cb).rank().to_numpy()) / (2 * n)
    raise ValueError(how)


In [ ]:
%%writefile /tmp/src/pair_extra.py
"""Targeted pair features for the error patterns of model v2 (step B). Country-agnostic, no labels.

Groups (column prefixes):
  b1_  descriptor-word difference: words in one name but not the other after removing legal words; how many are
       generic descriptors (directions, "services", "group", "holdings", ... in English and French); the
       IDF-weighted size of the difference (a rare extra word signals a different business).
  b2_  legal form different AND address the same / nearly the same (the "PLLC vs LLC at the same address" decoy).
  b3_  one address empty AND names (legal words removed) identical / near-identical, and how common the S1's core
       name is in its country / state (a rare name with an empty address is likely a match, a common one is not).
  b4_  trade-name case: same normalised address and same first house number but low name similarity; how rare the
       shared address is in the country (a rare shared address is strong evidence).
Counts (IDF, name frequency per country / state, address frequency per country) come from ALL train + test
records of each country, the same for every split.
"""
from __future__ import annotations

import math

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

B_GROUPS = {
    "b1": ["b1_n_diff", "b1_n_desc_diff", "b1_n_other_diff", "b1_idf_diff", "b1_idf_diff_max"],
    "b2": ["b2_legal_diff", "b2_legal_diff_addr_same", "b2_addr_sort"],
    "b3": ["b3_addr_empty_one", "b3_name_near", "b3_empty_near", "b3_name_freq_state", "b3_empty_near_rare"],
    "b4": ["b4_same_addr", "b4_same_num", "b4_trade", "b4_addr_freq", "b4_trade_rare_addr"],
}
DESCRIPTORS = {
    # directions / places
    "north", "south", "east", "west", "central", "northern", "southern", "eastern", "western", "upper", "lower",
    "nord", "sud", "est", "ouest", "grand", "greater", "new", "old", "city", "metro", "national", "international",
    "global", "india", "indian", "america", "american", "usa", "us", "france", "francais", "francaise",
    # generic business words
    "services", "service", "group", "groupe", "holdings", "holding", "center", "centre", "solutions", "solution",
    "enterprises", "enterprise", "industries", "industry", "associates", "associate", "partners", "brothers", "bros",
    "sons", "and", "company", "co", "trading", "traders", "systems", "technologies", "technology", "tech", "consulting",
    "consultants", "management", "ventures", "international", "the", "of", "de", "du", "des", "la", "le", "les", "et",
    "societe", "society", "agency", "agence", "store", "shop", "clinic", "hospital", "school", "ecole", "office",
    "branch", "unit", "division", "dept", "department", "main", "head", "hq",
}


def _tok(s: str | None) -> set[str]:
    return set(s.split()) if s else set()


def country_counts(records: pl.LazyFrame) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """records: ALL train + test records (S1, S2, S3) with country, state, core_name, addr_norm.
    Returns (token idf per country, core_name count per country + state, addr_norm count per country)."""
    r = records.select(pl.col("country").fill_null(""), pl.col("state").fill_null(""), pl.col("core_name").fill_null(""),
                       pl.col("addr_norm").fill_null(""))
    n = r.group_by("country").len("n_rec")
    idf = (r.select("country", pl.col("core_name").str.split(" ").list.unique().alias("t")).explode("t")
           .filter(pl.col("t").is_not_null() & (pl.col("t") != "")).group_by("country", "t").len("df")
           .join(n, on="country").select("country", "t", (pl.col("n_rec") / pl.col("df")).log().alias("idf")).collect())
    name_state = r.filter(pl.col("core_name") != "").group_by("country", "state", "core_name").len("n_name").collect()
    addr = r.filter(pl.col("addr_norm") != "").group_by("country", "addr_norm").len("n_addr").collect()
    return idf, name_state, addr


def extra_features(pairs: pl.DataFrame, s1_rec: pl.DataFrame, pool_rec: pl.DataFrame, idf: pl.DataFrame,
                   name_state: pl.DataFrame, addr: pl.DataFrame) -> pl.DataFrame:
    """pairs: s1, cand. *_rec: entity_id, country, state, core_name, nolegal, legal, addr_norm, numbers (prepared
    records, e.g. features.prepare_records). Returns s1, cand + every B_GROUPS column (float32), in pairs' order."""
    cols = ("country", "state", "core_name", "nolegal", "legal", "addr_norm", "numbers")
    x = (pairs.select("s1", "cand").with_row_index("_i")
         .join(s1_rec.select(pl.col("entity_id").alias("s1"), *[pl.col(c).alias(f"q_{c}") for c in cols]), on="s1", how="left")
         .join(pool_rec.select(pl.col("entity_id").alias("cand"), *[pl.col(c).alias(f"p_{c}") for c in cols]), on="cand", how="left")
         .sort("_i"))
    qn, pn = x["q_nolegal"].fill_null("").to_list(), x["p_nolegal"].fill_null("").to_list()
    country = x["q_country"].fill_null("").to_list()
    idf_map = {(c, t): v for c, t, v in idf.iter_rows()}
    idf_max = {c: g["idf"].max() for (c,), g in idf.group_by("country")}
    n_diff, n_desc, n_other, idf_sum, idf_mx = [], [], [], [], []
    for a, b, c in zip(qn, pn, country):
        d = _tok(a) ^ _tok(b)
        desc = [t for t in d if t in DESCRIPTORS]
        other = [t for t in d if t not in DESCRIPTORS]
        w = [idf_map.get((c, t), idf_max.get(c, 10.0)) for t in other]
        n_diff.append(len(d)); n_desc.append(len(desc)); n_other.append(len(other))
        idf_sum.append(sum(w)); idf_mx.append(max(w) if w else 0.0)
    near = process.cpdist(qn, pn, scorer=fuzz.token_sort_ratio, workers=-1) / 100.0
    qa, pa = x["q_addr_norm"].fill_null("").to_list(), x["p_addr_norm"].fill_null("").to_list()
    asort = process.cpdist(qa, pa, scorer=fuzz.token_sort_ratio, workers=-1) / 100.0
    empty_a = (x["q_addr_norm"].fill_null("") == "") | (x["p_addr_norm"].fill_null("") == "")
    ql, pl_ = x["q_legal"].fill_null(""), x["p_legal"].fill_null("")
    x = x.with_columns(
        pl.Series("b1_n_diff", n_diff), pl.Series("b1_n_desc_diff", n_desc), pl.Series("b1_n_other_diff", n_other),
        pl.Series("b1_idf_diff", idf_sum, dtype=pl.Float64), pl.Series("b1_idf_diff_max", idf_mx, dtype=pl.Float64),
        pl.Series("_near", near), pl.Series("_asort", np.where(empty_a.to_numpy(), np.nan, asort)),
        empty_a.alias("_empty"), ((ql != "") & (pl_ != "") & (ql != pl_)).alias("_legal_diff"),
        pl.col("q_numbers").list.first().alias("_qn1"), pl.col("p_numbers").list.first().alias("_pn1"),
    )
    x = (x.join(name_state.rename({"country": "q_country", "state": "q_state", "core_name": "q_core_name"}),
                on=["q_country", "q_state", "q_core_name"], how="left", maintain_order="left")
         .join(addr.rename({"country": "q_country", "addr_norm": "q_addr_norm"}), on=["q_country", "q_addr_norm"],
               how="left", maintain_order="left"))
    same_addr = (~pl.col("_empty")) & (pl.col("_asort") >= 0.95)
    same_num = (pl.col("_qn1").is_not_null() & (pl.col("_qn1") == pl.col("_pn1"))).fill_null(False)
    name_freq = pl.col("n_name").fill_null(0).cast(pl.Float64).log1p()
    addr_freq = pl.col("n_addr").fill_null(0).cast(pl.Float64).log1p()
    out = x.with_columns(
        pl.col("_legal_diff").cast(pl.Float32).alias("b2_legal_diff"),
        (pl.col("_legal_diff") & same_addr.fill_null(False)).cast(pl.Float32).alias("b2_legal_diff_addr_same"),
        pl.col("_asort").alias("b2_addr_sort"),
        pl.col("_empty").cast(pl.Float32).alias("b3_addr_empty_one"),
        (pl.col("_near") >= 0.95).cast(pl.Float32).alias("b3_name_near"),
        (pl.col("_empty") & (pl.col("_near") >= 0.95)).cast(pl.Float32).alias("b3_empty_near"),
        name_freq.alias("b3_name_freq_state"),
        (pl.col("_empty") & (pl.col("_near") >= 0.95) & (pl.col("n_name").fill_null(0) <= 3)).cast(pl.Float32).alias("b3_empty_near_rare"),
        same_addr.fill_null(False).cast(pl.Float32).alias("b4_same_addr"),
        same_num.cast(pl.Float32).alias("b4_same_num"),
        (same_addr.fill_null(False) & same_num & (pl.col("_near") < 0.5)).cast(pl.Float32).alias("b4_trade"),
        addr_freq.alias("b4_addr_freq"),
        (same_addr.fill_null(False) & same_num & (pl.col("_near") < 0.5) & (pl.col("n_addr").fill_null(0) <= 5)).cast(pl.Float32).alias("b4_trade_rare_addr"),
    )
    feats = [c for g in B_GROUPS.values() for c in g]
    return out.sort("_i").select("s1", "cand", *[pl.col(c).cast(pl.Float32) for c in feats])


In [ ]:
%%writefile /tmp/src/two_stage.py
"""Two-stage match model: stage-1 LightGBM (feature set C) -> set-level features -> stage-2 LightGBM.

Stage 1 gives every candidate pair a probability p1: out-of-fold on train (5 folds grouped by S1), the average
of the fold models on validation and test. Set features describe a pair relative to the other candidates of the
same S1 list (list-local, so they mean the same on train, validation and test):
  s_p1           stage-1 probability
  s_rank / s_gap rank of p1 in the S1's list, and the gap to the S1's best p1
  s_n05 / s_n09  number of candidates of the S1 with p1 > 0.5 / > 0.9
  s_sim_name / s_sim_addr / s_num_eq
                 the pair's candidate vs the S1's OTHER confident candidates (p1 > HIGH): max name token-set
                 ratio, max address token-set ratio, and whether any shares the first house number. A second
                 record of the same business looks like the first one; a decoy differs in number or legal form.
Cross-S1 counts (how many S1 lists a record is in) are deliberately NOT used: they depend on which S1 were
queried (300k of 2.2M train S1 vs all test S1), so they would shift between train and test. The reverse-search
rank / gap features already compare S1 for a record with all S1 competing.
"""
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

SEED = 42
HIGH = 0.8
N_FOLDS = 5
SET_FEATURES = ["s_p1", "s_rank", "s_gap", "s_n05", "s_n09", "s_sim_name", "s_sim_addr", "s_num_eq"]
PARAMS = dict(objective="binary", learning_rate=0.08, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=0)
ROUNDS, EARLY_STOP = 4000, 100


def fold_of(s1: pl.Series, n: int = N_FOLDS, salt: int = 0) -> np.ndarray:
    return (s1.hash(seed=SEED + salt) % n).cast(pl.Int64).to_numpy()


def fit(m: np.ndarray, y: np.ndarray, hold: np.ndarray, feats: list[str], seed: int = SEED, params: dict | None = None) -> lgb.Booster:
    p = {**PARAMS, **(params or {}), "seed": seed}
    dtr = lgb.Dataset(m[~hold], y[~hold], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(m[hold], y[hold], reference=dtr)
    return lgb.train(p, dtr, ROUNDS, valid_sets=[dva], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])


def predict(b: lgb.Booster, m: np.ndarray, chunk: int = 2_000_000) -> np.ndarray:
    return np.concatenate([b.predict(m[lo:lo + chunk], num_iteration=b.best_iteration) for lo in range(0, len(m), chunk)]
                          or [np.empty(0)])


def stage1_oof(x: pl.DataFrame, y: np.ndarray, feats: list[str], log=print) -> tuple[np.ndarray, list[lgb.Booster]]:
    """5-fold out-of-fold p1 on train (folds by S1); each fold model early-stops on a tenth of its own S1."""
    fold = fold_of(x["s1"])
    es = fold_of(x["s1"], 10, salt=1) == 0
    m = x.select(feats).to_numpy()
    oof, models = np.zeros(len(y)), []
    for k in range(N_FOLDS):
        tr = fold != k
        b = fit(m[tr], y[tr], es[tr], feats)
        oof[~tr] = predict(b, m[~tr])
        models.append(b)
        log(f"  stage-1 fold {k}: best iteration {b.best_iteration}")
    return oof, models


def stage1_predict(models: list[lgb.Booster], x: pl.DataFrame, feats: list[str]) -> np.ndarray:
    m = x.select(feats).to_numpy()
    return np.mean([predict(b, m) for b in models], axis=0)


def set_features(pairs: pl.DataFrame, p1: np.ndarray, rec: pl.DataFrame) -> pl.DataFrame:
    """pairs: s1, cand (all candidates of each S1 present); rec: entity_id, full_name, addr_norm, numbers (S2/S3
    records of the candidates). Returns s1, cand + SET_FEATURES, in the order of pairs."""
    x = pairs.select("s1", "cand").with_row_index("_i").with_columns(pl.Series("s_p1", p1, dtype=pl.Float64))
    x = x.with_columns(
        pl.col("s_p1").rank("ordinal", descending=True).over("s1").cast(pl.Float32).alias("s_rank"),
        (pl.col("s_p1").max().over("s1") - pl.col("s_p1")).alias("s_gap"),
        (pl.col("s_p1") > 0.5).sum().over("s1").cast(pl.Float32).alias("s_n05"),
        (pl.col("s_p1") > 0.9).sum().over("s1").cast(pl.Float32).alias("s_n09"),
    )
    r = rec.select(pl.col("entity_id").alias("cand"), pl.col("full_name").fill_null(""), pl.col("addr_norm").fill_null(""),
                   pl.col("numbers").list.first().alias("num1"))
    a = x.select("_i", "s1", "cand").join(r, on="cand", how="left")
    hi = x.filter(pl.col("s_p1") > HIGH).select("s1", pl.col("cand").alias("cand_o")).join(
        r.rename({"cand": "cand_o", "full_name": "name_o", "addr_norm": "addr_o", "num1": "num1_o"}), on="cand_o", how="left")
    pr = a.join(hi, on="s1", how="inner").filter(pl.col("cand") != pl.col("cand_o"))
    if pr.height:
        pr = pr.with_columns(
            pl.Series("sn", process.cpdist(pr["full_name"].to_list(), pr["name_o"].to_list(), scorer=fuzz.token_set_ratio, workers=-1) / 100.0),
            pl.Series("sa", process.cpdist(pr["addr_norm"].to_list(), pr["addr_o"].to_list(), scorer=fuzz.token_set_ratio, workers=-1) / 100.0),
            (pl.col("num1").is_not_null() & (pl.col("num1") == pl.col("num1_o"))).cast(pl.Float32).alias("ne"))
        agg = pr.group_by("_i").agg(pl.col("sn").max().alias("s_sim_name"), pl.col("sa").max().alias("s_sim_addr"),
                                    pl.col("ne").max().alias("s_num_eq"))
    else:
        agg = pl.DataFrame(schema={"_i": pl.UInt32, "s_sim_name": pl.Float64, "s_sim_addr": pl.Float64, "s_num_eq": pl.Float32})
    out = x.join(agg, on="_i", how="left").sort("_i")
    return out.select("s1", "cand", *[pl.col(c).cast(pl.Float32) for c in SET_FEATURES])


In [ ]:
%%writefile /tmp/src/model.py
"""Match model: LightGBM binary classifier on candidate pairs (features feat-v2, table 6).

Feature sets (the France question: French test candidates are found by fewer search methods, median 2 vs 4):
  A  all 86 features;
  B  A without f_n_methods (how many search methods found the pair);
  C  B without the pruner-derived features (f_p_u50, f_p_gap, f_list_rank): the blocking pruner itself uses the
     method count and the per-method ranks, so its score can carry the same bias.
Per-method scores and ranks stay in every set (they are similarities; "not found" is NaN).

Training S1 exclude the 100k train-split S1 the blocking pruner was trained on (their pruner scores and lists
are in-sample); `A_all` keeps them, to measure that effect. Early stopping uses a held-out tenth of the training
S1 (by S1, never by pair); validation S1 are never used for training or early stopping.
"""
from __future__ import annotations

import lightgbm as lgb
import numpy as np
import polars as pl

SEED = 42
PRUNER_DERIVED = ("f_p_u50", "f_p_gap", "f_list_rank")
PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100, feature_fraction=0.8,
              bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, seed=SEED, deterministic=True, verbose=-1,
              num_threads=0)
ROUNDS, EARLY_STOP = 3000, 100


def feature_sets(cols: list[str]) -> dict[str, list[str]]:
    f = [c for c in cols if c.startswith("f_")]
    b = [c for c in f if c != "f_n_methods"]
    return {"A": f, "B": b, "C": [c for c in b if c not in PRUNER_DERIVED]}


def early_stop_mask(s1: pl.Series, share: float = 0.1) -> np.ndarray:
    """True for rows whose S1 falls in the held-out share (stable hash of the id, same S1 -> same side)."""
    return ((s1.hash(seed=SEED) % 1000) < int(share * 1000)).to_numpy()


def train(x: pl.DataFrame, y: np.ndarray, feats: list[str], log=print) -> lgb.Booster:
    """x: s1 + feature columns; y: 0/1. Early stopping on a held-out tenth of the S1."""
    hold = early_stop_mask(x["s1"])
    m = x.select(feats).to_numpy()
    dtr = lgb.Dataset(m[~hold], y[~hold], feature_name=feats, free_raw_data=True)
    dva = lgb.Dataset(m[hold], y[hold], reference=dtr)
    booster = lgb.train(PARAMS, dtr, ROUNDS, valid_sets=[dva], valid_names=["early_stop"],
                        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])
    log(f"    {len(feats)} features, {int((~hold).sum()):,} fit rows, {int(hold.sum()):,} early-stop rows, "
        f"best iteration {booster.best_iteration}, early-stop logloss {booster.best_score['early_stop']['binary_logloss']:.4f}")
    return booster


def predict(booster: lgb.Booster, x: pl.DataFrame, chunk: int = 2_000_000) -> np.ndarray:
    feats = booster.feature_name()
    return np.concatenate([booster.predict(x[lo:lo + chunk].select(feats).to_numpy(), num_iteration=booster.best_iteration)
                           for lo in range(0, x.height, chunk)] or [np.empty(0)])


def importance(booster: lgb.Booster, name: str) -> pl.DataFrame:
    g = booster.feature_importance("gain")
    return (pl.DataFrame({"variant": name, "feature": booster.feature_name(), "gain": g.astype(float)})
            .with_columns((pl.col("gain") / pl.col("gain").sum()).alias("gain_share")).sort("gain", descending=True))


In [ ]:
%%writefile /tmp/src/features.py
"""Pair features for the main model (stream A -> B, table 6 in the README).

One row per candidate pair (s1, cand), in the input order, with columns s1, cand, f_* (float32, NaN = missing).
No label, country or language column: France exists only in test, so every feature is country-agnostic. The
record's country is used only to pick the legal-word list (as normalisation does) and as the key of the counts
and state maps (as blocking does). The one source feature is f_cand_is_s3 (S2 vs S3 candidate).

Inputs:
- pairs: the candidate table (table 4/5): s1, cand, score_*/rank_* per blocking method, best_* (optional; else
  computed over the given list), p_u50 (optional). Missing blocking columns give all-NaN features, so the schema
  is the same for every split.
- s1_rec / pool_rec: normalised records (table 1) of the S1 and S2/S3 side, with missing states filled in
  (fill_record_states), passed through prepare_records().
- stats: SideStats, counts over ALL train + test records (S1, S2 and S3) of each country, no labels. The same
  object serves every split, so a count means the same in train, val and test.

Groups:
1. name: similarities on full_name, core_name and `nolegal` (legal words removed at ANY position, the agreed fix
   for "Willow LLC Center"); first/last token, extra tokens, IDF-weighted overlap, rarest differing token.
2. rare words: core-name tokens that are rare in the country (document frequency <= RARE_DF and 3+ characters,
   the definition of blocking's rare-token search): shared, only on one side, and how rare the rarest shared one is.
3. legal form: agreement of the edge legal form (`legal`) and of legal words found anywhere in the name.
4. address: fuzzy similarities; city and state agreement (missing states filled with blocking's maps: city ->
   state, then dept -> state; on the real data no dept -> state map passes blocking's thresholds, 0 in every
   country, so in practice missing states come from the city only); city found in the other address; number
   tokens (shared, conflicting, first/last). No département or postcode feature: on real data they almost never
   fire (features-v2-sample: French S1 never has a département; 5-6 digit postcodes are in ~0.1% of addresses,
   most such numbers being US house numbers).
5. blocking: scores, ranks, gaps to the S1's best, number of methods, pruner score and list position. The rare
   method's score/rank are left out (98% NaN, no signal); the rare-word group replaces them.
6. context: name frequency in the country; the pair's value minus the best OTHER candidate of the same S1;
   duplicates of the candidate's name in the S1's list. Every context feature looks only inside one S1's list:
   nothing counts how many S1 lists a candidate appears in (that number depends on which S1 were queried).
7. source: f_cand_is_s3.
"""
from __future__ import annotations

import math
from collections.abc import Iterable
from dataclasses import dataclass

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

from blocking import RARE_DF, fill_states, state_maps
from normalise import _legal_for

FEATURE_VERSION = "feat-v2"
METHODS = ("name", "name_city", "name_addr", "reverse", "rare")
SCORED = ("name", "name_city", "name_addr", "reverse")  # rare score/rank dropped in v2 (see group 5)
FORWARD = ("name", "name_city", "name_addr")
REC_COLS = ("entity_id", "country", "full_name", "core_name", "legal", "addr_norm", "city", "state", "numbers")
TRAILING_ONLY = {"pra", "li"}  # Indian legal spellings that are legal only as a trailing token (normalisation rule)
NAME_VARIANTS = ("full", "core", "nolegal")
RARE_MIN_LEN = 3  # blocking.rare_tokens: rare tokens have df <= RARE_DF and at least 3 characters
CONTEXT = ("f_name_core_tsr", "f_name_core_ratio", "f_name_idf_jacc", "f_addr_tsr", "f_addr_c3_jacc", "f_num_jacc")


# ---------------------------------------------------------------------------- records, states and counts

def legal_words(country: str | None) -> dict[str, str]:
    """Legal word -> canonical form for this country, for removal at any position (trailing-only forms excluded)."""
    table = _legal_for((country or "").strip().lower())
    return {w: c for w, c in table.items() if w not in TRAILING_ONLY}


def country_state_maps(loc: pl.DataFrame) -> dict[str, dict[str, pl.DataFrame]]:
    """Per country, blocking's city -> state and dept -> state maps. loc: country, city, dept, state of ALL train +
    test records (S1, S2, S3), no labels: the same maps blocking used for its search buckets."""
    return {c: state_maps(g) for (c,), g in loc.group_by("country")}


def fill_record_states(rec: pl.DataFrame, maps: dict[str, dict[str, pl.DataFrame]]) -> pl.DataFrame:
    """Missing states filled from city, then département, per country (blocking.fill_states; on the real data the
    département map is empty, so only the city fills states). Row order is not kept."""
    parts = [fill_states(g, maps[c]) if c in maps else g for (c,), g in rec.group_by("country")]
    return pl.concat(parts, how="diagonal_relaxed").drop("state_src", strict=False) if parts else rec


def prepare_records(rec: pl.DataFrame) -> pl.DataFrame:
    """Normalised records -> the columns used here, nulls filled, plus `nolegal` (full_name without legal words at
    any position; falls back to core_name) and `legal_any` (sorted canonical legal words found anywhere)."""
    rec = rec.select(REC_COLS).with_columns(
        *[pl.col(c).fill_null("") for c in ("country", "full_name", "core_name", "legal", "addr_norm")],
        pl.col("numbers").fill_null([]),
    )
    key = pl.col("country").str.to_lowercase().str.strip_chars()
    parts = []
    for k, g in rec.with_columns(key.alias("_k")).group_by("_k", maintain_order=True):
        table = legal_words(k[0])
        words = list(table)
        toks = pl.col("full_name").str.split(" ")
        parts.append(g.with_columns(
            toks.list.eval(pl.element().filter(~pl.element().is_in(words))).list.join(" ").alias("nolegal"),
            toks.list.eval(pl.element().filter(pl.element().is_in(words)).replace_strict(table, default=None))
            .list.unique().list.sort().list.join(" ").alias("legal_any"),
        ))
    out = pl.concat(parts) if parts else rec.with_columns(pl.lit("").alias("nolegal"), pl.lit("").alias("legal_any"), pl.lit("").alias("_k"))
    return out.with_columns(pl.when(pl.col("nolegal") == "").then(pl.col("core_name")).otherwise(pl.col("nolegal")).alias("nolegal")).drop("_k")


@dataclass
class SideStats:
    """Counts over ALL train + test records of each country (S1, S2 and S3), no labels. Keyed by the country string
    exactly as blocking keys its per-country token counts."""
    n: dict[str, int]  # country -> number of records
    token_df: pl.DataFrame  # country, t, df: records whose core_name contains token t
    name_n: pl.DataFrame  # country, core_name, n_s1, n_pool: S1 / S2+S3 records with this core_name


def count_stats(s1: Iterable[pl.DataFrame | pl.LazyFrame], pool: Iterable[pl.DataFrame | pl.LazyFrame]) -> SideStats:
    """s1 / pool: frames (or lazy scans, one per file) with country and core_name of ALL S1 / ALL S2+S3 records of
    train AND test. Pass the same frames for every split."""
    names, toks = [], []
    for col, frames in (("n_s1", s1), ("n_pool", pool)):
        for f in frames:
            f = f.lazy().select(pl.col("country").fill_null(""), pl.col("core_name").fill_null(""))
            names.append(f.group_by("country", "core_name").agg(pl.len().alias(col)).collect())
            toks.append(f.select("country", pl.col("core_name").str.split(" ").list.unique().alias("t"))
                        .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != ""))
                        .group_by("country", "t").agg(pl.len().alias("df")).collect())
    empty = {"country": pl.String, "core_name": pl.String, "n_s1": pl.UInt32, "n_pool": pl.UInt32}
    name_n = (pl.concat(names, how="diagonal_relaxed") if names else pl.DataFrame(schema=empty))
    for c in ("n_s1", "n_pool"):
        if c not in name_n.columns:
            name_n = name_n.with_columns(pl.lit(0, pl.UInt32).alias(c))
    name_n = name_n.group_by("country", "core_name").agg(pl.col("n_s1").sum(), pl.col("n_pool").sum())
    token_df = (pl.concat(toks).group_by("country", "t").agg(pl.col("df").sum()) if toks
                else pl.DataFrame(schema={"country": pl.String, "t": pl.String, "df": pl.UInt32}))
    n = name_n.group_by("country").agg((pl.col("n_s1") + pl.col("n_pool")).sum().alias("n"))
    return SideStats(dict(zip(n["country"].to_list(), n["n"].to_list())), token_df, name_n)


# ---------------------------------------------------------------------------- scalar helpers

def _f32(name: str, a) -> pl.Series:
    return pl.Series(name, np.asarray(a, dtype=np.float32)).fill_nan(None)


def _grams(s: str, n: int) -> set[str]:
    if n == 0:
        return set(s.split())
    s = f" {s} "
    return {s[i:i + n] for i in range(len(s) - n + 1)} if len(s) >= n else {s}


def _jaccard(a: list[str], b: list[str], n: int = 0) -> np.ndarray:
    """Token (n=0) or character n-gram Jaccard; NaN when either side is empty."""
    out = np.full(len(a), np.nan, np.float32)
    for i, (s, t) in enumerate(zip(a, b)):
        if s and t:
            A, B = _grams(s, n), _grams(t, n)
            out[i] = len(A & B) / len(A | B)
    return out


def _fuzzy(scorer, a: list[str], b: list[str]) -> np.ndarray:
    """rapidfuzz scorer in 0..1 over aligned pairs; NaN when either side is empty."""
    s = process.cpdist(a, b, scorer=scorer, workers=-1).astype(np.float32)
    if scorer is not JaroWinkler.normalized_similarity:
        s /= 100.0
    empty = (np.array([len(x) for x in a]) == 0) | (np.array([len(x) for x in b]) == 0)
    s[empty] = np.nan
    return s


def _eq(a: list, b: list) -> np.ndarray:
    """1 = same, 0 = different, NaN = missing on either side (null or empty)."""
    return np.array([np.nan if not x or not y else float(x == y) for x, y in zip(a, b)], np.float32)


def _df_lookup(stats: SideStats, country: list[str], names: list[str]) -> dict[tuple[str, str], int]:
    """(country, token) -> document frequency, for the tokens of the given names only."""
    t = (pl.DataFrame({"country": country, "t": pl.Series(names, dtype=pl.String).str.split(" ")})
         .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != "")).unique()
         .join(stats.token_df, on=["country", "t"], how="left"))
    return dict(zip(zip(t["country"].to_list(), t["t"].to_list()), t["df"].fill_null(0).to_list()))


# ---------------------------------------------------------------------------- feature groups (row level)

def _name_features(q: dict, p: dict, country: list[str], stats: SideStats) -> list[pl.Series]:
    out = []
    for v, col in zip(NAME_VARIANTS, ("full_name", "core_name", "nolegal")):
        a, b = q[col], p[col]
        out += [_f32(f"f_name_{v}_ratio", _fuzzy(fuzz.ratio, a, b)),
                _f32(f"f_name_{v}_tsr", _fuzzy(fuzz.token_set_ratio, a, b)),
                _f32(f"f_name_{v}_tsort", _fuzzy(fuzz.token_sort_ratio, a, b)),
                _f32(f"f_name_{v}_jw", _fuzzy(JaroWinkler.normalized_similarity, a, b)),
                _f32(f"f_name_{v}_tok_jacc", _jaccard(a, b, 0)),
                _f32(f"f_name_{v}_c3_jacc", _jaccard(a, b, 3))]
    a, b = q["core_name"], p["core_name"]
    out.append(_f32("f_name_core_partial", _fuzzy(fuzz.partial_ratio, a, b)))
    dfl = _df_lookup(stats, country + country, a + b)
    n = len(a)
    first, last, only_q, only_p, len_diff, contains, idf_j, idf_max = (np.full(n, np.nan, np.float32) for _ in range(8))
    r_shared, r_only_q, r_only_p, r_idf = (np.full(n, np.nan, np.float32) for _ in range(4))
    for i, (s, t, c) in enumerate(zip(a, b, country)):
        if not (s and t):
            continue
        ts, tt = s.split(), t.split()
        A, B = set(ts), set(tt)
        first[i], last[i] = ts[0] == tt[0], ts[-1] == tt[-1]
        only_q[i], only_p[i] = len(A - B), len(B - A)
        len_diff[i] = abs(len(s) - len(t))
        contains[i] = f" {s} " in f" {t} " or f" {t} " in f" {s} "
        df = {x: dfl.get((c, x), 0) for x in A | B}
        n_c = stats.n.get(c, 0)
        w_unknown = math.log(n_c + 1) or 1.0
        w = {x: math.log((n_c + 1) / (d + 1)) for x, d in df.items()}
        union = sum(w.values())
        idf_j[i] = sum(w[x] for x in A & B) / union if union > 0 else 1.0
        diff = [w[x] for x in A ^ B]
        idf_max[i] = max(diff) / w_unknown if diff else 0.0
        rare = {x for x, d in df.items() if d <= RARE_DF and len(x) >= RARE_MIN_LEN}
        shared = rare & A & B
        r_shared[i], r_only_q[i], r_only_p[i] = len(shared), len((rare & A) - B), len((rare & B) - A)
        r_idf[i] = max(w[x] for x in shared) / w_unknown if shared else 0.0
    out += [_f32("f_name_first_eq", first), _f32("f_name_last_eq", last), _f32("f_name_only_s1", only_q),
            _f32("f_name_only_cand", only_p), _f32("f_name_len_diff", len_diff), _f32("f_name_contains", contains),
            _f32("f_name_idf_jacc", idf_j), _f32("f_name_idf_max_diff", idf_max),
            _f32("f_rare_shared", r_shared), _f32("f_rare_only_s1", r_only_q), _f32("f_rare_only_cand", r_only_p),
            _f32("f_rare_shared_idf", r_idf)]
    return out


def _legal_features(q: dict, p: dict) -> list[pl.Series]:
    out = []
    for tag, col in (("", "legal"), ("_any", "legal_any")):
        a, b = q[col], p[col]
        ea, eb = np.array([not x for x in a]), np.array([not x for x in b])
        same = np.array([x == y for x, y in zip(a, b)], np.float32)
        same[ea | eb] = np.nan
        out += [_f32(f"f_legal{tag}_both_missing", ea & eb), _f32(f"f_legal{tag}_one_missing", ea ^ eb),
                _f32(f"f_legal{tag}_same", same)]
    out.append(_f32("f_legal_tok_jacc", _jaccard(q["legal"], p["legal"], 0)))
    return out


def _address_features(q: dict, p: dict) -> list[pl.Series]:
    a = [s.replace(",", " ").replace("  ", " ") for s in q["addr_norm"]]
    b = [s.replace(",", " ").replace("  ", " ") for s in p["addr_norm"]]
    qc, pc = q["city"], p["city"]
    city_fz = _fuzzy(fuzz.ratio, [c or "" for c in qc], [c or "" for c in pc])
    city_in = np.full(len(a), np.nan, np.float32)
    for i, (c1, c2, s, t) in enumerate(zip(qc, pc, a, b)):
        hits = [f" {c} " in f" {addr} " for c, addr in ((c1, t), (c2, s)) if c and addr]
        if hits:
            city_in[i] = any(hits)
    out = [_f32("f_addr_missing", [not s or not t for s, t in zip(a, b)]),
           _f32("f_addr_ratio", _fuzzy(fuzz.ratio, a, b)),
           _f32("f_addr_tsr", _fuzzy(fuzz.token_set_ratio, a, b)),
           _f32("f_addr_tsort", _fuzzy(fuzz.token_sort_ratio, a, b)),
           _f32("f_addr_partial", _fuzzy(fuzz.partial_ratio, a, b)),
           _f32("f_addr_tok_jacc", _jaccard(a, b, 0)),
           _f32("f_addr_c3_jacc", _jaccard(a, b, 3)),
           _f32("f_city_same", _eq(qc, pc)), _f32("f_city_ratio", city_fz), _f32("f_city_in_addr", city_in),
           _f32("f_state_agree", _eq(q["state"], p["state"]))]
    n = len(a)
    cols = {k: np.full(n, np.nan, np.float32) for k in
            ("n_s1", "n_cand", "shared", "only_s1", "only_cand", "jacc", "first_eq", "last_eq")}
    for i, (x, y) in enumerate(zip(q["numbers"], p["numbers"])):
        x, y = list(x or []), list(y or [])
        A, B = set(x), set(y)
        cols["n_s1"][i], cols["n_cand"][i] = len(x), len(y)
        if not (x and y):
            continue
        cols["shared"][i], cols["only_s1"][i], cols["only_cand"][i] = len(A & B), len(A - B), len(B - A)
        cols["jacc"][i] = len(A & B) / len(A | B)
        cols["first_eq"][i], cols["last_eq"][i] = x[0] == y[0], x[-1] == y[-1]
    return out + [_f32(f"f_num_{k}", v) for k, v in cols.items()]


def _blocking_features(x: pl.DataFrame) -> list[pl.Expr]:
    """From the candidate table; absent columns give all-NaN features so every split has the same schema."""
    has = set(x.columns)
    col = lambda c: pl.col(c).cast(pl.Float32) if c in has else pl.lit(None, dtype=pl.Float32)
    best = lambda m: col(f"best_{m}") if f"best_{m}" in has else col(f"score_{m}").max().over("s1")
    out = [*[col(f"score_{m}").alias(f"f_score_{m}") for m in SCORED],
           *[col(f"rank_{m}").alias(f"f_rank_{m}") for m in SCORED],
           *[(best(m) - col(f"score_{m}")).alias(f"f_gap_{m}") for m in FORWARD],
           (col("best_reverse") - col("score_reverse")).alias("f_gap_reverse"),
           pl.sum_horizontal(*[col(f"rank_{m}").is_not_null() for m in METHODS]).cast(pl.Float32).alias("f_n_methods"),
           col("p_u50").alias("f_p_u50"),
           (col("p_u50").max().over("s1") - col("p_u50")).alias("f_p_gap"),
           col("p_u50").rank("ordinal", descending=True).over("s1").cast(pl.Float32).alias("f_list_rank"),
           pl.len().over("s1").cast(pl.Float32).alias("f_list_size"),
           pl.col("cand").str.starts_with("S3-").cast(pl.Float32).alias("f_cand_is_s3")]
    return out


# ---------------------------------------------------------------------------- assembly

def row_features(pairs: pl.DataFrame, s1_rec: pl.DataFrame, pool_rec: pl.DataFrame, stats: SideStats) -> pl.DataFrame:
    """Per-pair features that need only the pair itself (safe to compute in row chunks). Keeps pairs' order and
    carries q_core / p_core for the context step."""
    cols = ("country", "full_name", "core_name", "nolegal", "legal", "legal_any", "addr_norm", "city", "state", "numbers")
    x = (pairs.select("s1", "cand").with_row_index("_i")
         .join(s1_rec.select(pl.col("entity_id").alias("s1"), *[pl.col(c).alias(f"q_{c}") for c in cols]), on="s1", how="left")
         .join(pool_rec.select(pl.col("entity_id").alias("cand"), *[pl.col(c).alias(f"p_{c}") for c in cols]), on="cand", how="left")
         .sort("_i"))
    missing = x.filter(pl.col("q_core_name").is_null() | pl.col("p_core_name").is_null()).height
    if missing:
        raise ValueError(f"{missing} pairs have no normalised record for s1 or cand")
    q = {c: x[f"q_{c}"].to_list() for c in cols}
    p = {c: x[f"p_{c}"].to_list() for c in cols}
    feats = [*_name_features(q, p, q["country"], stats), *_legal_features(q, p), *_address_features(q, p)]
    names = stats.name_n.select("country", "core_name", "n_s1", "n_pool")
    return (x.select("s1", "cand", pl.col("q_core_name").alias("q_core"), pl.col("p_core_name").alias("p_core"),
                     "q_country", "p_country")
            .with_columns(*feats)
            .join(names.rename({"country": "q_country", "core_name": "q_core", "n_s1": "_s1n", "n_pool": "_pq"}),
                  on=["q_country", "q_core"], how="left", maintain_order="left")
            .join(names.select(pl.col("country").alias("p_country"), pl.col("core_name").alias("p_core"), pl.col("n_pool").alias("_pp")),
                  on=["p_country", "p_core"], how="left", maintain_order="left")
            .with_columns(*[pl.col(c).fill_null(0).cast(pl.Float32).log1p().alias(f)
                            for c, f in (("_s1n", "f_freq_s1_name_in_s1"), ("_pq", "f_freq_s1_name_in_pool"),
                                         ("_pp", "f_freq_cand_name_in_pool"))])
            .drop("_s1n", "_pq", "_pp", "q_country", "p_country"))


def context_features(x: pl.DataFrame) -> pl.DataFrame:
    """Needs every candidate of an S1 in x: value minus the best OTHER candidate of the same S1 (null when the list
    has one candidate), and how many other candidates share this candidate's core name. Only within one S1's list."""
    x = x.with_row_index("_i")
    top = x.group_by("s1").agg(*[pl.col(c).drop_nulls().top_k(2).alias(f"_top_{c}") for c in CONTEXT])
    x = x.join(top, on="s1", how="left").sort("_i")
    other = lambda c: (pl.when(pl.col(c) == pl.col(f"_top_{c}").list.get(0, null_on_oob=True))
                       .then(pl.col(f"_top_{c}").list.get(1, null_on_oob=True))
                       .otherwise(pl.col(f"_top_{c}").list.get(0, null_on_oob=True)))
    return x.with_columns(
        *[(pl.col(c) - other(c)).alias(f"f_ctx_{c[2:]}") for c in CONTEXT],
        (pl.len().over("s1", "p_core") - 1).cast(pl.Float32).alias("f_ctx_same_name_in_list"),
        (pl.col("f_name_core_tsr") >= 1.0).sum().over("s1").cast(pl.Float32).alias("f_ctx_n_name_exact"),
    ).drop("_i", *[f"_top_{c}" for c in CONTEXT])


def build(pairs: pl.DataFrame, s1_rec: pl.DataFrame, pool_rec: pl.DataFrame, stats: SideStats,
          chunk: int = 1_000_000) -> pl.DataFrame:
    """Table 6 for one split: s1, cand, f_* (float32, NaN = missing), rows in the order of `pairs`.
    `pairs` must hold every candidate of each S1 it contains (context and list features are per S1)."""
    dup = pairs.select("s1", "cand").is_duplicated().sum()
    if dup:
        raise ValueError(f"{dup} duplicated (s1, cand) rows")
    blocking = pairs.select("s1", "cand", *_blocking_features(pairs))
    rows = pl.concat([row_features(pairs[lo:lo + chunk], s1_rec, pool_rec, stats) for lo in range(0, pairs.height, chunk)]
                     or [row_features(pairs.clear(), s1_rec, pool_rec, stats)])
    x = context_features(rows).drop("q_core", "p_core")
    assert x.select("s1", "cand").equals(blocking.select("s1", "cand")), "row order changed"
    x = x.hstack(blocking.drop("s1", "cand"))
    f = [c for c in x.columns if c.startswith("f_")]
    return x.select("s1", "cand", *[pl.col(c).cast(pl.Float32).fill_null(float("nan")) for c in f])


def feature_names(x: pl.DataFrame) -> list[str]:
    return [c for c in x.columns if c.startswith("f_")]


In [ ]:
%%writefile /tmp/src/normalise.py
"""Shared normalisation for names and addresses (used by blocking, features and inference).

Deterministic and vectorised with polars; the only per-string Python call is anyascii
transliteration, applied once per distinct non-ASCII string.

Open-set countries: rules are looked up by the lower-cased country label. Country tables only ADD
cleaning for known countries; any other label (new in test, empty, made up) gets the generic rules
(transliteration, cleaning, generic legal suffixes and abbreviations, number tokens, city guess).

Names   -> full_name (legal suffix canonicalised), core_name (legal suffix removed), legal (the suffix)
Address -> addr_norm, city, state, dept (French departement), numbers (digit tokens, leading zeros stripped)
Fallbacks: core_name falls back to full_name; text fields are empty only when the raw value has no
letters or digits. city / state / dept are null when not found. normalise_records() keeps the raw columns.

What is learned from data (see docs/normalisation.md):
- state aliases: from TRAIN true pairs only (labels), excluding validation S1; saved to a versioned file.
- city frequency vocabulary: from train + test records (no labels), per country.
"""
from __future__ import annotations

import json
import re
from collections.abc import Mapping
from pathlib import Path

import polars as pl
from anyascii import anyascii

# Bump whenever a rule or list changes; stored in every output so cached artifacts can be checked.
NORM_VERSION = "norm-v3"  # v2: international legal forms only for France / unknown countries; v3: cie, Malayalam/Gurmukhi ltd/pvt, no dangling "and"

# ---------------------------------------------------------------------------- basic cleaning


def transliterate(s: pl.Series) -> pl.Series:
    """anyascii for non-ASCII strings (Devanagari, Tamil, accents, ...); ASCII strings untouched."""
    s = s.cast(pl.String).fill_null("")
    mask = s.str.contains(r"[^\x00-\x7F]")
    uniq = s.filter(mask).unique().to_list()
    if not uniq:
        return s
    return s.replace({u: anyascii(u) for u in uniq})


def _clean(e: pl.Expr, drop_dots: bool) -> pl.Expr:
    """Lowercase, & -> and, punctuation -> space, collapse whitespace. For names, dots and
    apostrophes are deleted (l.l.c. -> llc, nash's -> nashs); for addresses they become spaces
    (no.330 -> no 330)."""
    e = e.str.to_lowercase().str.replace_all("&", " and ")
    if drop_dots:
        e = e.str.replace_all(r"[.'`]", "")
    return e.str.replace_all(r"[^a-z0-9]+", " ").str.strip_chars()


def clean_text(text: str, drop_dots: bool = False) -> str:
    """Scalar version of transliterate + _clean (for building lookup tables and tests)."""
    s = anyascii(text).lower().replace("&", " and ")
    if drop_dots:
        s = re.sub(r"[.'`]", "", s)
    return " ".join(re.findall(r"[a-z0-9]+", s))


def _key(country: str | None) -> str:
    return (country or "").strip().lower()


def _ckey(countries: pl.Series) -> pl.Series:
    return countries.cast(pl.String).fill_null("").str.to_lowercase().str.strip_chars()


# ---------------------------------------------------------------------------- name rules (language knowledge)

# Legal suffixes: token -> canonical token. Only a trailing run (or, for LEGAL_LEADING, a leading run)
# of these is treated as a legal form; the same words mid-name are kept.
LEGAL_GENERAL = {  # all countries
    "inc": "inc", "incorporated": "inc", "incorporation": "inc",
    "llc": "llc", "llp": "llp", "lp": "lp", "plc": "plc", "pllc": "pllc", "pc": "pc", "psc": "psc",
    "corp": "corp", "corporation": "corp", "co": "co", "company": "co", "cos": "co",
    "ltd": "ltd", "limited": "ltd", "pvt": "pvt", "private": "pvt", "opc": "opc",
}
# Continental-European forms. Applied to France and to every country WITHOUT its own table (open set),
# but not to the US or India, where "spa" / "sas" / "sl" are ordinary words ("Ramey Spa Inc", "Sas Nagar").
LEGAL_INTERNATIONAL = {
    "gmbh": "gmbh", "bv": "bv", "srl": "srl", "spa": "spa", "sl": "sl", "slu": "slu",
    "sarl": "sarl", "sas": "sas", "sasu": "sasu", "eurl": "eurl", "selarl": "selarl", "eirl": "eirl", "scop": "scop",
    "cie": "co",  # "& Cie" = "& Co"
}
LEGAL_BY_COUNTRY = {
    "us": {},
    "india": {
        # transliterations seen in S2/S3 (Devanagari / Tamil / Kannada / Odia -> anyascii), e.g. "pra li" = "pvt ltd"
        "praivet": "pvt", "praibhet": "pvt", "piraivet": "pvt", "praivett": "pvt", "prayvet": "pvt", "pra": "pvt",
        "praivrr": "pvt",  # Malayalam
        "limitet": "ltd", "limited": "ltd", "limitedd": "ltd", "li": "ltd", "elelpi": "llp", "ellpi": "llp",
        "limirrd": "ltd",  # Malayalam
        "limtid": "ltd",  # Gurmukhi
    },
    "france": {"sa": "sa", "sci": "sci", "snc": "snc", "ei": "ei", "sca": "sca", "scp": "scp", "gie": "gie", "scm": "scm"},
}
# Legal forms that also appear as a LEADING token after word reordering ("LLC Value Electronics").
# Only unambiguous abbreviations: never words like "company" or "limited", or short forms like "co"/"sa".
LEGAL_LEADING = {"inc", "llc", "llp", "ltd", "pvt", "corp", "plc", "pllc"}
LEGAL_LEADING_INTERNATIONAL = {"gmbh", "sarl", "sas", "sasu", "eurl", "snc"}
KNOWN_ENGLISH_ONLY = {"us", "india"}  # countries whose tables replace the international forms


def _legal_for(country: str) -> dict[str, str]:
    extra = {} if country in KNOWN_ENGLISH_ONLY else LEGAL_INTERNATIONAL
    return {**LEGAL_GENERAL, **extra, **LEGAL_BY_COUNTRY.get(country, {})}


def _leading_for(country: str) -> set[str]:
    return LEGAL_LEADING if country in KNOWN_ENGLISH_ONLY else LEGAL_LEADING | LEGAL_LEADING_INTERNATIONAL
# multi-token spellings, collapsed before suffix detection (anchored at the end of the name)
LEGAL_MULTI = [(r" l l c$", " llc"), (r" l l p$", " llp"), (r" p l l c$", " pllc"), (r" l p$", " lp"), (r" p c$", " pc")]

# Honorific prefixes (repeated prefixes are all removed): India only, where they are noise added to names.
HONORIFICS_BY_COUNTRY = {
    "india": ["m s", "ms", "messrs", "smt", "shrimati", "shri", "shree", "sri", "sree", "kumari", "km", "mr", "mrs", "dr"],
}

# Word abbreviations (token -> expansion)
NAME_ABBREV_GENERAL = {
    "intl": "international", "mfg": "manufacturing", "mfrs": "manufacturers",
    "svc": "services", "svcs": "services", "srvcs": "services", "mgmt": "management", "mgt": "management",
    "assn": "association", "assoc": "associates", "dept": "department", "natl": "national",
    "govt": "government", "univ": "university", "hosp": "hospital", "ctr": "center", "cntr": "center",
    "centre": "center", "bros": "brothers", "engg": "engineering", "engr": "engineering", "grp": "group",
    "hldgs": "holdings", "inds": "industries", "sys": "systems", "solns": "solutions",
}
NAME_ABBREV_BY_COUNTRY = {"france": {"et": "and", "st": "saint", "ste": "sainte"}}


# ---------------------------------------------------------------------------- address rules (language knowledge)

ADDR_ABBREV_GENERAL = {
    "rd": "road", "ave": "avenue", "blvd": "boulevard", "hwy": "highway", "pkwy": "parkway",
    "bldg": "building", "apt": "apartment", "flr": "floor",
}
ADDR_ABBREV_BY_COUNTRY = {
    "us": {
        "st": "street", "str": "street", "dr": "drive", "ln": "lane", "ct": "court", "cir": "circle", "pl": "place",
        "sq": "square", "ste": "suite", "fl": "floor", "ter": "terrace", "terr": "terrace", "trl": "trail",
        "pt": "point", "mt": "mount", "ft": "fort", "hts": "heights", "rte": "route", "fwy": "freeway",
        "expy": "expressway", "cres": "crescent",
    },
    "india": {"nr": "near", "opp": "opposite", "fl": "floor", "ngr": "nagar", "bldg": "building"},
    "france": {
        "st": "saint", "ste": "sainte", "av": "avenue", "bd": "boulevard", "bld": "boulevard", "bvd": "boulevard",
        "boul": "boulevard", "rte": "route", "chem": "chemin", "pl": "place", "imp": "impasse", "fbg": "faubourg",
        "sq": "square", "all": "allee", "r": "rue",
    },
}
LANDMARK_PREFIX = r"^(near|nr|opp|opposite|behind|beside|next to|adjacent to|in front of)\b"
COUNTRY_WORDS = {"india", "bharat", "usa", "us", "united states", "united states of america", "america", "france",
                 "null", "none", "na", "n a", "nil"}
CITY_PREFIX = r"^(city of|town of|village of|township of|borough of|ville de)\s+"
CITY_SUFFIX = r"\s+(city|township|town|village|borough|cdp)$"
# A component "<postcode> <words>" or "<words> <postcode>" (4-6 digits) is a city candidate once the
# postcode is removed, unless the words look like a street (then the number is a house number) or a
# unit / box ("po box 4823", "suite 1200", "pmb 12345").
POSTCODE_EDGE = r"^\d{4,6}\s+|\s+\d{4,6}$"
STREET_WORDS = (r"\b(road|street|avenue|lane|drive|boulevard|highway|parkway|court|circle|trail|terrace|way|"
                r"rue|allee|impasse|chemin|route|quai|calle|avenida|carrer|via|viale|piazza|marg|floor|building|"
                r"po|box|pmb|unit|suite|ste|apt|apartment|room|rm|flat|plot|no|door|house|block|office|shop|sector|"
                r"ward|lot|survey|khasra|gali|lane)\b"
                r"|(strasse|gasse|weg|platz)\b")

US_STATES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california", "co": "colorado",
    "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia", "hi": "hawaii", "id": "idaho",
    "il": "illinois", "in": "indiana", "ia": "iowa", "ks": "kansas", "ky": "kentucky", "la": "louisiana",
    "me": "maine", "md": "maryland", "ma": "massachusetts", "mi": "michigan", "mn": "minnesota",
    "ms": "mississippi", "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada",
    "nh": "new hampshire", "nj": "new jersey", "nm": "new mexico", "ny": "new york", "nc": "north carolina",
    "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon", "pa": "pennsylvania",
    "ri": "rhode island", "sc": "south carolina", "sd": "south dakota", "tn": "tennessee", "tx": "texas",
    "ut": "utah", "vt": "vermont", "va": "virginia", "wa": "washington", "wv": "west virginia",
    "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia", "pr": "puerto rico", "gu": "guam",
    "vi": "virgin islands", "as": "american samoa", "mp": "northern mariana islands",
}
# canonical = full state name; value = codes and alternative spellings
INDIA_STATES = {
    "andhra pradesh": ["ap"], "arunachal pradesh": ["ar"], "assam": ["as"], "bihar": ["br"],
    "chhattisgarh": ["cg", "ct", "chattisgarh"], "goa": ["ga"], "gujarat": ["gj", "gujrat"],
    "haryana": ["hr"], "himachal pradesh": ["hp"], "jharkhand": ["jh"], "karnataka": ["ka"],
    "kerala": ["kl", "keralam"], "madhya pradesh": ["mp"], "maharashtra": ["mh"], "manipur": ["mn"],
    "meghalaya": ["ml"], "mizoram": ["mz"], "nagaland": ["nl"], "odisha": ["od", "or", "orissa"],
    "punjab": ["pb"], "rajasthan": ["rj"], "sikkim": ["sk"], "tamil nadu": ["tn", "tamilnadu"],
    "telangana": ["tg", "ts"], "tripura": ["tr"], "uttar pradesh": ["up"],
    "uttarakhand": ["uk", "ua", "ut", "uttaranchal"], "west bengal": ["wb"],
    "andaman and nicobar islands": ["an", "andaman and nicobar"], "chandigarh": ["ch"],
    "dadra and nagar haveli and daman and diu": ["dn", "dd", "dadra and nagar haveli", "daman and diu"],
    "delhi": ["dl", "nct of delhi", "national capital territory of delhi"], "jammu and kashmir": ["jk"],
    "ladakh": ["la"], "lakshadweep": ["ld"], "puducherry": ["py", "pondicherry"],
}
# canonical = current region; value = old regions / alternative spellings
FRANCE_REGIONS = {
    "auvergne rhone alpes": ["auvergne", "rhone alpes"], "bourgogne franche comte": ["bourgogne", "franche comte"],
    "bretagne": ["brittany"], "centre val de loire": ["centre"], "corse": ["corsica"],
    "grand est": ["alsace", "lorraine", "champagne ardenne"],
    "hauts de france": ["nord pas de calais", "picardie"], "ile de france": [],
    "normandie": ["basse normandie", "haute normandie", "normandy"],
    "nouvelle aquitaine": ["aquitaine", "limousin", "poitou charentes"],
    "occitanie": ["languedoc roussillon", "midi pyrenees"], "pays de la loire": [],
    "provence alpes cote d azur": ["paca", "provence alpes cote dazur"],
    "guadeloupe": [], "martinique": [], "guyane": ["french guiana"], "la reunion": ["reunion"], "mayotte": [],
}
# Departements (Paris omitted: as an address component it is the city)
FRANCE_DEPTS = [
    "ain", "aisne", "allier", "alpes de haute provence", "hautes alpes", "alpes maritimes", "ardeche", "ardennes",
    "ariege", "aube", "aude", "aveyron", "bouches du rhone", "calvados", "cantal", "charente", "charente maritime",
    "cher", "correze", "corse du sud", "haute corse", "cote d or", "cotes d armor", "creuse", "dordogne", "doubs",
    "drome", "eure", "eure et loir", "finistere", "gard", "haute garonne", "gers", "gironde", "herault",
    "ille et vilaine", "indre", "indre et loire", "isere", "jura", "landes", "loir et cher", "loire",
    "haute loire", "loire atlantique", "loiret", "lot", "lot et garonne", "lozere", "maine et loire", "manche",
    "marne", "haute marne", "mayenne", "meurthe et moselle", "meuse", "morbihan", "moselle", "nievre", "nord",
    "oise", "orne", "pas de calais", "puy de dome", "pyrenees atlantiques", "hautes pyrenees",
    "pyrenees orientales", "bas rhin", "haut rhin", "rhone", "haute saone", "saone et loire", "sarthe",
    "savoie", "haute savoie", "seine maritime", "seine et marne", "yvelines", "deux sevres", "somme", "tarn",
    "tarn et garonne", "var", "vaucluse", "vendee", "vienne", "haute vienne", "vosges", "yonne",
    "territoire de belfort", "essonne", "hauts de seine", "seine saint denis", "val de marne", "val d oise",
]


def _state_table(country: str) -> dict[str, str]:
    """Hand-written variant -> canonical state/region, keys cleaned like address components.
    Empty for any country without a table (open set)."""
    t: dict[str, str] = {}
    if country == "us":
        for code, name in US_STATES.items():
            t[code] = code
            t[clean_text(name)] = code
    elif country == "india":
        for name, alts in INDIA_STATES.items():
            t[clean_text(name)] = name
            for a in alts:
                t[clean_text(a)] = name
    elif country == "france":
        for name, alts in FRANCE_REGIONS.items():
            t[clean_text(name)] = name
            for a in alts:
                t[clean_text(a)] = name
    return t


def _dept_table(country: str) -> dict[str, str]:
    return {clean_text(d): clean_text(d) for d in FRANCE_DEPTS} if country == "france" else {}


# ---------------------------------------------------------------------------- normaliser


class Normaliser:
    """Holds the lookup tables. `state_aliases` ({country: {variant: canonical}}) are learned from
    training pairs with learn_state_aliases() and added on top of the hand-written tables."""

    def __init__(self, state_aliases: Mapping[str, Mapping[str, str]] | None = None, version: str = NORM_VERSION):
        self.state_aliases = {_key(c): dict(m) for c, m in (state_aliases or {}).items()}
        self.version = version

    @classmethod
    def load(cls, path: str | Path | None) -> "Normaliser":
        """Load aliases saved by save_state_aliases(); a missing path gives the hand-written tables only."""
        if not path or not Path(path).exists():
            return cls()
        data = json.loads(Path(path).read_text())
        if "aliases" in data:  # versioned file
            if data.get("norm_version") != NORM_VERSION:
                raise ValueError(f"{path} was learned with {data.get('norm_version')}, code is {NORM_VERSION}")
            return cls(data["aliases"])
        return cls(data)

    def states(self, country: str) -> dict[str, str]:
        return {**_state_table(country), **self.state_aliases.get(country, {})}

    # ------------------------------------------------------------------ names
    def names(self, names: pl.Series, countries: pl.Series) -> pl.DataFrame:
        raw = names.cast(pl.String).fill_null("")
        df = pl.DataFrame({"n": transliterate(raw), "ckey": _ckey(countries)})
        df = df.with_row_index("rid").with_columns(_clean(pl.col("n"), drop_dots=True).alias("n"))
        parts = [self._names_one(g, ckey) for (ckey,), g in df.group_by("ckey")]
        if not parts:
            return pl.DataFrame(schema={"full_name": pl.String, "core_name": pl.String, "legal": pl.String})
        return pl.concat(parts).sort("rid").drop("rid")

    def _names_one(self, g: pl.DataFrame, country: str) -> pl.DataFrame:
        e = pl.col("n")
        hon = HONORIFICS_BY_COUNTRY.get(country)
        if hon:
            stripped = e.str.replace(r"^(?:(?:" + "|".join(hon) + r")\s+)+", "")
            e = pl.when(stripped != "").then(stripped).otherwise(e)
        abbrev = {**NAME_ABBREV_GENERAL, **NAME_ABBREV_BY_COUNTRY.get(country, {})}
        e = e.str.split(" ").list.eval(pl.element().replace(abbrev)).list.join(" ")
        for pat, rep in LEGAL_MULTI:
            e = e.str.replace(pat, rep)
        legal = _legal_for(country)
        alt = "|".join(sorted(map(re.escape, legal), key=len, reverse=True))
        tail_pat = r"^(.*?)((?: (?:" + alt + r"))*)$"
        lead_pat = r"^((?:(?:" + "|".join(sorted(_leading_for(country))) + r") )+)(.+)$"
        g = g.with_columns(e.alias("_e")).with_columns(
            pl.col("_e").str.extract(lead_pat, 1).fill_null("").alias("_lead"),
            pl.coalesce(pl.col("_e").str.extract(lead_pat, 2), pl.col("_e")).alias("_rest"),
        )
        g = g.with_columns((pl.lit(" ") + pl.col("_rest")).alias("_s"))
        core = pl.col("_s").str.extract(tail_pat, 1).fill_null("").str.strip_chars()
        g = g.with_columns(
            core.alias("core_name"),
            pl.concat_str([pl.col("_lead"), pl.col("_s").str.extract(tail_pat, 2).fill_null("")], separator=" ")
            .str.replace_all(r"\s+", " ").str.strip_chars().str.split(" ")
            .list.eval(pl.element().replace(legal)).list.unique(maintain_order=True)  # "pvt ltd ltd" -> "pvt ltd"
            .list.join(" ").str.strip_chars().alias("legal"),
        )
        # "Smith & Co", "Elsa & Cie SARL", "Dupont et Cie": the connector belongs to the legal form
        g = g.with_columns(
            pl.when(pl.col("legal") != "").then(pl.col("core_name").str.replace(r"(?:\s+and)+$", ""))
            .otherwise(pl.col("core_name")).alias("core_name"))
        full = pl.concat_str([pl.col("core_name"), pl.col("legal")], separator=" ").str.strip_chars()
        g = g.with_columns(full.alias("full_name"))
        # Fallbacks: a name that is only legal tokens ("LLC", "Pvt Ltd") keeps them as its core.
        only_legal = pl.col("core_name") == ""
        return g.select(
            "rid",
            "full_name",
            pl.when(only_legal).then(pl.col("full_name")).otherwise(pl.col("core_name")).alias("core_name"),
            pl.when(only_legal).then(pl.lit("")).otherwise(pl.col("legal")).alias("legal"),
        )

    # ------------------------------------------------------------------ addresses
    def addresses(self, addrs: pl.Series, countries: pl.Series, city_vocab: pl.DataFrame | None = None) -> pl.DataFrame:
        """city_vocab: (ckey, cand, freq) from city_counts(); when None it is fitted on these addresses.
        The city is the most frequent city-like component, so reordered addresses agree."""
        raw = addrs.cast(pl.String).fill_null("")
        df = pl.DataFrame({"a": transliterate(raw), "ckey": _ckey(countries)}).with_row_index("rid")
        comps = self._components(df)
        vocab = city_vocab if city_vocab is not None else _vocab(comps)
        cities = (
            comps.filter(pl.col("kind") == "cand")
            .join(vocab, on=["ckey", "cand"], how="left")
            .group_by("rid")
            .agg(pl.col("cand").sort_by([pl.col("freq").fill_null(0), pl.col("pos")], descending=[True, True]).first().alias("city"))
        )
        per = comps.group_by("rid").agg(
            pl.col("c").sort_by("pos").str.join(", ").alias("addr_norm"),  # keeps component boundaries (idempotent)
            pl.col("state").drop_nulls().last().alias("state"),
            pl.col("dept").drop_nulls().last().alias("dept"),
        )
        out = (
            df.select("rid", "a")
            .join(per, on="rid", how="left")
            .join(cities, on="rid", how="left")
            .with_columns(
                pl.col("addr_norm").fill_null(""),
                _clean(pl.col("a"), drop_dots=False).str.extract_all(r"\d+")
                .list.eval(pl.element().str.replace(r"^0+(\d)", "$1")).list.unique(maintain_order=True).alias("numbers"),
            )
        )
        return out.sort("rid").select("addr_norm", "city", "state", "dept", "numbers")

    def _components(self, df: pl.DataFrame) -> pl.DataFrame:
        """One row per non-empty comma component: rid, ckey, pos, c (normalised), state, dept, kind, cand."""
        comps = (
            df.select("rid", "ckey", pl.col("a").str.split(",").alias("c"))
            .with_columns(pl.int_ranges(pl.col("c").list.len()).alias("pos"))
            .explode("c", "pos", empty_as_null=True)
            .with_columns(_clean(pl.col("c"), drop_dots=False).alias("c"))
            .filter(pl.col("c").is_not_null() & (pl.col("c") != ""))
        )
        parts = []
        for (ckey,), g in comps.group_by("ckey"):
            abbrev = {**ADDR_ABBREV_GENERAL, **ADDR_ABBREV_BY_COUNTRY.get(ckey, {})}
            # States first, on the cleaned component: US codes like "ct", "fl", "mt" would otherwise be
            # expanded to court / floor / mount. A state may carry a trailing postcode ("nc 27260").
            bare = pl.col("c").str.replace(r"(\s\d{5,6})+$", "")
            g = g.with_columns(
                bare.replace_strict(self.states(ckey), default=None, return_dtype=pl.String).alias("state"),
                bare.replace_strict(_dept_table(ckey), default=None, return_dtype=pl.String).alias("dept"),
            )
            expanded = pl.col("c").str.split(" ").list.eval(pl.element().replace(abbrev)).list.join(" ")
            g = g.with_columns(pl.coalesce(pl.col("state"), pl.col("dept"), expanded).alias("c"))
            parts.append(g)
        schema = {"rid": pl.UInt32, "ckey": pl.String, "c": pl.String, "pos": pl.Int64, "state": pl.String, "dept": pl.String}
        comps = pl.concat(parts, how="vertical_relaxed") if parts else pl.DataFrame(schema=schema)
        no_post = pl.col("c").str.replace_all(POSTCODE_EDGE, "").str.strip_chars()
        city_like = (no_post != "") & ~no_post.str.contains(r"\d") & ~no_post.str.contains(STREET_WORDS)
        stripped = no_post.str.replace(CITY_PREFIX, "").str.replace(CITY_SUFFIX, "")
        cand = pl.when(stripped != "").then(stripped).otherwise(no_post)
        kind = (
            pl.when(pl.col("state").is_not_null()).then(pl.lit("state"))
            .when(pl.col("dept").is_not_null()).then(pl.lit("dept"))
            .when(pl.col("c").str.contains(LANDMARK_PREFIX)).then(pl.lit("landmark"))
            .when(pl.col("c").is_in(list(COUNTRY_WORDS))).then(pl.lit("country"))
            .when(~pl.col("c").str.contains(r"\d")).then(pl.lit("cand"))
            .when(city_like).then(pl.lit("cand"))  # "10115 berlin", "paris 75001"
            .otherwise(pl.lit("number"))
        )
        return comps.with_columns(kind.alias("kind"), cand.alias("cand"))

    def city_counts(self, addrs: pl.Series, countries: pl.Series) -> pl.DataFrame:
        """(ckey, cand, freq) for one batch of addresses. Sum over batches (train + test, all sources,
        no labels) with merge_city_counts() to build the vocabulary passed to addresses()."""
        df = pl.DataFrame({"a": transliterate(addrs.cast(pl.String).fill_null("")), "ckey": _ckey(countries)}).with_row_index("rid")
        return _vocab(self._components(df))


def _vocab(comps: pl.DataFrame) -> pl.DataFrame:
    return comps.filter(pl.col("kind") == "cand").group_by("ckey", "cand").agg(pl.len().cast(pl.Int64).alias("freq"))


def merge_city_counts(parts: list[pl.DataFrame]) -> pl.DataFrame:
    return pl.concat(parts).group_by("ckey", "cand").agg(pl.col("freq").sum())


def normalise_records(df: pl.DataFrame, norm: Normaliser, city_vocab: pl.DataFrame | None = None,
                      name_col: str = "business_name", addr_col: str = "business_address",
                      country_col: str = "country") -> pl.DataFrame:
    """All original columns (raw name/address kept) + normalised name and address fields + norm_version."""
    n = norm.names(df[name_col], df[country_col])
    a = norm.addresses(df[addr_col], df[country_col], city_vocab)
    return pl.concat([df, n, a], how="horizontal_extend").with_columns(pl.lit(norm.version).alias("norm_version"))


# ---------------------------------------------------------------------------- learned state aliases


def learn_state_aliases(pairs: pl.DataFrame, min_count: int = 30, purity: float = 0.9,
                        max_s1_ratio: float = 0.02, max_city_concentration: float = 1.5,
                        max_with_state: float = 0.2) -> dict[str, dict[str, str]]:
    """Learn state spellings from TRAIN true pairs (columns: country, addr_1 (S1), addr_2 (match)).
    Uses labels: never pass pairs of validation S1 entities.

    A match-side city-like component becomes an alias of state X when
    - it co-occurs with S1 state X in >= `purity` of its >= `min_count` pairs,
    - it is (almost) never an S1 component (<= `max_s1_ratio`): keeps real cities such as "mumbai" out,
    - its pairs do not concentrate on one S1 city much more than the state's pairs do overall
      (<= `max_city_concentration` x): keeps old city names such as "bombay" / "poona" out,
    - it REPLACES the state: <= `max_with_state` of the match addresses containing it also contain a
      known state component. Keeps city abbreviations such as "atl" (written "ATL, GA") out."""
    base = Normaliser()
    ckey = _ckey(pairs["country"])
    s1c = base._components(pl.DataFrame({"a": transliterate(pairs["addr_1"]), "ckey": ckey}).with_row_index("rid"))
    s1_state = s1c.filter(pl.col("state").is_not_null()).group_by("rid").agg(pl.col("state").last().alias("s1_state"))
    s1_city = base.addresses(pairs["addr_1"], pairs["country"]).select("city").with_row_index("rid")
    s1_state = s1_state.join(s1_city, on="rid", how="left")
    s1_freq = s1c.group_by("ckey", "c").agg(pl.len().alias("n_s1"))
    mc = base._components(pl.DataFrame({"a": transliterate(pairs["addr_2"]), "ckey": ckey}).with_row_index("rid"))
    has_state = mc.group_by("rid").agg((pl.col("kind") == "state").any().alias("has_state"))
    unknown = mc.filter(pl.col("kind") == "cand").select("rid", "ckey", "c").unique().join(has_state, on="rid")
    co = unknown.join(s1_state, on="rid")
    per_c = co.group_by("ckey", "c").agg(pl.col("has_state").mean().alias("with_state_share"))
    city_share = (
        co.group_by("ckey", "c", "city").agg(pl.len().alias("nc"))
        .group_by("ckey", "c").agg((pl.col("nc").max() / pl.col("nc").sum()).alias("top_city_share"))
    )
    state_ckey = s1_state.join(s1c.select("rid", "ckey").unique("rid"), on="rid")
    state_share = (
        state_ckey.group_by("ckey", "s1_state", "city").agg(pl.len().alias("nc"))
        .group_by("ckey", "s1_state").agg((pl.col("nc").max() / pl.col("nc").sum()).alias("state_top_city_share"))
    )
    stats = (
        co.group_by("ckey", "c", "s1_state").agg(pl.len().alias("n"))
        .with_columns(pl.col("n").sum().over("ckey", "c").alias("n_c"))
        .join(s1_freq, on=["ckey", "c"], how="left")
        .join(city_share, on=["ckey", "c"], how="left")
        .join(state_share, on=["ckey", "s1_state"], how="left")
        .join(per_c, on=["ckey", "c"], how="left")
        .with_columns(pl.col("n_s1").fill_null(0))
        .filter((pl.col("n_c") >= min_count) & (pl.col("n") >= purity * pl.col("n_c"))
                & (pl.col("n_s1") <= max_s1_ratio * pl.col("n_c"))
                & (pl.col("top_city_share") <= max_city_concentration * pl.col("state_top_city_share"))
                & (pl.col("with_state_share") <= max_with_state))
    )
    out: dict[str, dict[str, str]] = {}
    for ck, c, st in stats.select("ckey", "c", "s1_state").sort("ckey", "c").iter_rows():
        out.setdefault(ck, {})[c] = st
    return out


def alias_diagnostics(pairs: pl.DataFrame, aliases: Mapping[str, Mapping[str, str]]) -> pl.DataFrame:
    """Evidence per learned alias (pair count, purity) for review."""
    ckey = _ckey(pairs["country"])
    base = Normaliser()
    s1c = base._components(pl.DataFrame({"a": transliterate(pairs["addr_1"]), "ckey": ckey}).with_row_index("rid"))
    s1_state = s1c.filter(pl.col("state").is_not_null()).group_by("rid").agg(pl.col("state").last().alias("s1_state"))
    mc = base._components(pl.DataFrame({"a": transliterate(pairs["addr_2"]), "ckey": ckey}).with_row_index("rid"))
    rows = [{"ckey": ck, "c": a, "state": s} for ck, m in aliases.items() for a, s in m.items()]
    if not rows:
        return pl.DataFrame(schema={"ckey": pl.String, "alias": pl.String, "state": pl.String, "pairs": pl.UInt32, "purity": pl.Float64})
    al = pl.DataFrame(rows)
    hits = mc.select("rid", "ckey", "c").unique().join(al, on=["ckey", "c"]).join(s1_state, on="rid", how="left")
    return (
        hits.group_by("ckey", "c", "state")
        .agg(pl.len().alias("pairs"), (pl.col("s1_state") == pl.col("state")).mean().alias("purity"))
        .rename({"c": "alias"})
        .sort("ckey", "pairs", descending=[False, True])
    )


def save_state_aliases(aliases: Mapping[str, Mapping[str, str]], path: str | Path, meta: Mapping | None = None) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    data = {"norm_version": NORM_VERSION, "meta": dict(meta or {}), "aliases": aliases}
    Path(path).write_text(json.dumps(data, indent=1, sort_keys=True, ensure_ascii=False))


In [ ]:
%%writefile /tmp/src/blocking.py
"""Candidate generation (blocking): union of several searches, per country.

Methods (all per country; the country is a compute split, 0 true pairs cross countries):
- name       forward TF-IDF top-k on core_name
- name_city  forward TF-IDF top-k on core_name + city
- name_addr  forward TF-IDF top-k on core_name + normalised address
- reverse    for every S2/S3 record, its top-m S1 on core_name + address (all S1 of the split compete); the
             record's best score is kept so a margin rule (keep rank 2 only if close to rank 1) can be applied
- rare       S2/S3 records sharing a rare core_name token (document frequency <= RARE_DF in the country)

TF-IDF: char_wb 3-4-grams, fitted per country on train + test records (no labels; a seeded sample
of up to FIT_SAMPLE texts). N-grams in more than MAX_DF of the texts are dropped: they carry almost no
weight and dominate the cost of the sparse product.

Search scope (a compute split, not a filter):
- "state": a forward query searches the pool records of its own state plus all pool records without a
  state; an S1 without a state searches the whole country. The reverse search runs within states; a pool
  record without a state searches all S1 of the country (fallback). Before bucketing, a missing state is
  inferred from the city, then the dept (state_maps / fill_states, learned from records that have both). On the
  challenge data no dept -> state map passes the rule, so in practice only the city fills states.
- "country": every search covers the whole country (affordable on a GPU); recovers pairs whose states
  disagree.
The rare-token index ignores states in both scopes.

Top-k search runs on the GPU when torch sees one (dense score blocks + torch.topk), otherwise on CPU with
sparse_dot_topn. check_gpu() compares the two on a slice before a run (tie-aware: scores per rank and
exact recomputation) and the faster device does the searches.

Near-identical decoys are NOT filtered here; every method keeps its full top-k.
"""
from __future__ import annotations

import time
from collections.abc import Callable

import numpy as np
import polars as pl
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

SEED = 42
FIT_SAMPLE = 3_000_000
MAX_DF = 0.02
RARE_DF = 20
NGRAMS = (3, 4)
FORWARD = ("name", "name_city", "name_addr")
REVERSE_REP = "name_addr"
NONE = "__none__"
ALL = "__all__"
SCOPES = ("state", "country")
INFER_MIN_N, INFER_MIN_SHARE = 20, 0.9  # state inference from city / dept (bucket placement only)
N_THREADS = 4


def texts(df: pl.DataFrame) -> pl.DataFrame:
    """Text representations used by the searches (all from norm-v3 columns)."""
    core = pl.col("core_name")
    return df.select(
        core.alias("name"),
        pl.concat_str([core, pl.col("city").fill_null("")], separator=" ").str.strip_chars().alias("name_city"),
        pl.concat_str([core, pl.col("addr_norm").str.replace_all(",", "")], separator=" ").str.strip_chars().alias("name_addr"),
    )


def fit_vectorizer(corpus: pl.Series) -> TfidfVectorizer:
    if corpus.len() > FIT_SAMPLE:
        corpus = corpus.sample(FIT_SAMPLE, seed=SEED)
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=NGRAMS, min_df=2, max_df=MAX_DF,
                        sublinear_tf=True, dtype=np.float32)
    v.fit(corpus.to_list())
    return v


try:  # GPU path (Kaggle GPU); falls back to sparse_dot_topn on CPU
    import torch
    USE_GPU = torch.cuda.is_available()
except ImportError:
    torch, USE_GPU = None, False
GPU_BLOCK_BYTES = 2 * 1024**3  # cap for each dense block per step (scores: chunk x pool; queries: chunk x vocab)


def gpu_info() -> str:
    if not USE_GPU:
        return "no GPU (CPU sparse_dot_topn)"
    return f"torch {torch.__version__}, {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB"


def _to_torch_csr(M: sparse.csr_matrix, device: str):
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int32)), torch.from_numpy(M.indices.astype(np.int32)),
                                   torch.from_numpy(M.data.astype(np.float32)), size=M.shape, device=device)


def _topn_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None, device: str = "cuda"):
    """Top-k of Q @ P.T per row on the GPU: P stays on the card as CSR; each query chunk is densified,
    multiplied (cuSPARSE SpMM) and torch.topk keeps the best k pool rows per query."""
    Pg = _to_torch_csr(P, device)
    chunk = int(max(1, min(4096, GPU_BLOCK_BYTES // (4 * P.shape[0]), GPU_BLOCK_BYTES // (4 * Q.shape[1]))))
    rows, cols, vals, ranks = [], [], [], []
    for s in range(0, Q.shape[0], chunk):
        Qc = _to_torch_csr(Q[s:s + chunk], device).to_dense()      # chunk x V
        scores = torch.sparse.mm(Pg, Qc.T).T.contiguous()           # chunk x pool
        v, i = torch.topk(scores, k, dim=1)                         # chunk x k, sorted descending
        v, i = v.cpu().numpy().reshape(-1), i.cpu().numpy().reshape(-1).astype(np.int64)
        n = scores.shape[0]
        r, rk = np.repeat(np.arange(s, s + n), k), np.tile(np.arange(1, k + 1), n)
        keep = v > 0  # as on CPU: only pairs sharing an n-gram
        if keep_col is not None:
            keep &= keep_col[i]
        rows.append(r[keep]); cols.append(i[keep]); vals.append(v[keep]); ranks.append(rk[keep])
        del Qc, scores
    del Pg
    return tuple(np.concatenate(x) for x in (rows, cols, vals, ranks))


def _topn_cpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    R = sp_matmul_topn(Q, P.T.tocsr(), top_n=k, threshold=0.0, sort=True, n_threads=N_THREADS).tocsr()
    counts = np.diff(R.indptr)
    rows = np.repeat(np.arange(R.shape[0]), counts)
    ranks = np.arange(R.nnz) - np.repeat(R.indptr[:-1], counts) + 1
    cols = R.indices.astype(np.int64)
    keep = np.ones(R.nnz, bool) if keep_col is None else keep_col[cols]
    return rows[keep], cols[keep], R.data.astype(np.float32)[keep], ranks[keep]


def _topn(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, keep_col: np.ndarray | None = None):
    """Top-k columns of Q @ P.T per row: (row, col, score, rank 1..k). keep_col: optional mask over P rows;
    pairs whose pool row is not kept are dropped after ranking (ranks are unchanged)."""
    if Q.shape[0] == 0 or P.shape[0] == 0:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.float32), e
    k = min(k, P.shape[0])
    return (_topn_gpu if USE_GPU else _topn_cpu)(Q, P, k, keep_col)


def check_gpu(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int, n_q: int = 4000, n_p: int = 1_000_000) -> dict:
    """GPU vs CPU top-k on a slice, tie-aware. Many pool records share a name, so equal scores at the k-th
    place may be kept differently; instead of comparing (row, col) pairs, compare the score at every rank
    (rank_score_diff) and recompute each GPU pair's cosine exactly on CPU (pair_score_diff). Timing runs after
    a warm-up call, so CUDA start-up is not counted."""
    Q, P = Q[:n_q], P[:n_p]
    k = min(k, P.shape[0])
    _topn_gpu(Q[:64], P, k); torch.cuda.synchronize()  # warm-up (CUDA context, cuSPARSE handles)
    t0 = time.time(); g = _topn_gpu(Q, P, k); torch.cuda.synchronize(); t1 = time.time()
    c = _topn_cpu(Q, P, k); t2 = time.time()
    by_rank = lambda x: pl.DataFrame({"r": x[0], "rk": x[3], "v": x[2]})
    j = by_rank(c).join(by_rank(g), on=["r", "rk"], how="full", suffix="_g").with_columns(pl.col("v", "v_g").fill_null(0.0))
    exact = np.asarray(Q[g[0]].multiply(P[g[1]]).sum(axis=1)).ravel()
    gp, cp = set(zip(g[0].tolist(), g[1].tolist())), set(zip(c[0].tolist(), c[1].tolist()))
    pairs = Q.shape[0] * P.shape[0]
    return {"queries": Q.shape[0], "pool": P.shape[0], "k": k,
            "rank_score_diff": float((j["v"] - j["v_g"]).abs().max() or 0.0),
            "pair_score_diff": float(np.abs(exact - g[2]).max()) if len(exact) else 0.0,
            "pair_agreement": len(gp & cp) / max(len(cp), 1),
            "gpu_ns_per_pair": (t1 - t0) / pairs * 1e9, "cpu_ns_per_pair": (t2 - t1) / pairs * 1e9}


def use_gpu(flag: bool) -> None:
    """Choose the device for all later searches (GPU only if CUDA is available)."""
    global USE_GPU
    USE_GPU = bool(flag) and torch is not None and torch.cuda.is_available()


def state_maps(records: pl.DataFrame, min_n: int = INFER_MIN_N, min_share: float = INFER_MIN_SHARE) -> dict[str, pl.DataFrame]:
    """Learn key -> state from records that have both (no labels): for city and for dept, the most frequent state
    when it covers >= min_share of min_n+ records. Used only to place records without a detected state into a
    search bucket (a compute split), never as a filter or a feature."""
    out = {}
    for key in ("city", "dept"):
        if key not in records.columns:
            continue
        c = (records.filter(pl.col(key).is_not_null() & pl.col("state").is_not_null())
             .group_by(key, "state").len()
             .with_columns(pl.col("len").sum().over(key).alias("tot"))
             .sort(key, "len", "state", descending=[False, True, False])
             .group_by(key, maintain_order=True).first())
        out[key] = c.filter((pl.col("tot") >= min_n) & (pl.col("len") / pl.col("tot") >= min_share)).select(
            key, pl.col("state").alias(f"state_from_{key}"))
    return out


def fill_states(df: pl.DataFrame, maps: dict[str, pl.DataFrame]) -> pl.DataFrame:
    """state_src = detected / city / dept / none; state = detected, else inferred from city, else from dept."""
    x = df
    for key, m in maps.items():
        x = x.join(m, on=key, how="left")
    inferred = [pl.col(f"state_from_{k}") for k in maps]
    src = pl.when(pl.col("state").is_not_null()).then(pl.lit("detected"))
    for k in maps:
        src = src.when(pl.col(f"state_from_{k}").is_not_null()).then(pl.lit(k))
    return x.with_columns(src.otherwise(pl.lit("none")).alias("state_src"),
                          pl.coalesce(pl.col("state"), *inferred).alias("state")).drop([f"state_from_{k}" for k in maps])


def buckets(s1: pl.DataFrame, pool: pl.DataFrame, scope: str) -> tuple[np.ndarray, np.ndarray]:
    if scope == "country":
        return np.full(s1.height, ALL, dtype=object), np.full(pool.height, ALL, dtype=object)
    return s1["state"].fill_null(NONE).to_numpy(), pool["state"].fill_null(NONE).to_numpy()


def forward(Q, q_bucket: np.ndarray, P, p_bucket: np.ndarray, k: int) -> pl.DataFrame:
    """For each query row: top-k pool rows among its state bucket + pool rows without a state."""
    out = []
    p_none = np.flatnonzero(p_bucket == NONE)
    for b in np.unique(q_bucket):
        qi = np.flatnonzero(q_bucket == b)
        pi = np.arange(P.shape[0]) if b == NONE else np.concatenate([np.flatnonzero(p_bucket == b), p_none])
        r, c, s, rk = _topn(Q[qi], P[pi], k)
        out.append(pl.DataFrame({"qi": qi[r], "pi": pi[c], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def reverse(S, s_bucket: np.ndarray, P, p_bucket: np.ndarray, m: int, keep_s: np.ndarray | None = None,
            none_fallback: bool = True) -> pl.DataFrame:
    """For each pool row: its top-m S1 rows in the same state (pool rows without a state: over all S1 of the
    country if none_fallback, else skipped). Returns (qi = S1 row, pi, score, rank,
    best), where best is the pool row's top-1 score over ALL S1 (for margin rules). keep_s: optional mask over S1
    rows; only pairs with a kept S1 are returned (all S1 still compete)."""
    out = []
    for b in np.unique(p_bucket):
        if b == NONE and not none_fallback:
            continue
        pi = np.flatnonzero(p_bucket == b)
        si = np.arange(S.shape[0]) if b == NONE else np.flatnonzero(s_bucket == b)  # no state: whole country
        r, c, s, rk = _topn(P[pi], S[si], m)
        best = np.zeros(len(pi), dtype=np.float32)
        best[r[rk == 1]] = s[rk == 1]
        f = slice(None) if keep_s is None else keep_s[si[c]]
        out.append(pl.DataFrame({"qi": si[c][f], "pi": pi[r][f], "score": s[f], "rank": rk[f], "best": best[r][f]}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32,
                                                           "rank": pl.Int64, "best": pl.Float32})


def rare_tokens(q_names: pl.Series, p_names: pl.Series, df_counts: pl.DataFrame, max_df: int = RARE_DF) -> pl.DataFrame:
    """Pool rows sharing a rare core_name token with the query (token df <= max_df in the country, so at most
    ~max_df x tokens rows per query). score = number of shared rare tokens; rank by score, ties by pool row
    (deterministic, so pieces merge exactly)."""
    rare = df_counts.filter((pl.col("df") <= max_df) & (pl.col("t").str.len_chars() >= 3)).select("t")
    tok = lambda s, name: (pl.DataFrame({name: np.arange(s.len()), "t": s.str.split(" ")})
                           .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null()).unique()
                           .join(rare, on="t"))
    qt, pt = tok(q_names, "qi"), tok(p_names, "pi")
    hits = qt.join(pt, on="t").group_by("qi", "pi").agg(pl.len().cast(pl.Float32).alias("score"))
    return (hits.sort("qi", "score", "pi", descending=[False, True, False])
            .with_columns(pl.int_range(1, pl.len() + 1, dtype=pl.Int64).over("qi").alias("rank"))
            .select("qi", "pi", "score", "rank"))


def token_df(names: pl.Series) -> pl.DataFrame:
    """Document frequency of core_name tokens (distinct per record)."""
    return (pl.DataFrame({"i": np.arange(names.len()), "t": names.str.split(" ")})
            .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != "")).unique()
            .group_by("t").agg(pl.len().alias("df")))


def block_country(s1: pl.DataFrame, pool: pl.DataFrame, query_mask: np.ndarray, fit_corpus: dict[str, pl.Series],
                  df_counts: pl.DataFrame, k: int, m: int, scopes: tuple[str, ...] = ("state",),
                  skip: Callable[[str, str, float], bool] = lambda scope, method, pairs: False,
                  log: Callable[[str], None] = print, gpu_check: bool = False,
                  timing: list[dict] | None = None, pool_mask: np.ndarray | None = None,
                  reverse_keep: np.ndarray | None = None) -> tuple[pl.DataFrame, list[dict]]:
    """All methods for one country, for each search scope (same TF-IDF matrices). s1: ALL S1 of the split
    (reverse competition); query_mask: S1 rows to generate candidates for. skip(scope, method, pairs) may
    drop a search (e.g. over a time budget). Returns a long table (scope, qi = S1 row, pi = pool row,
    method, score, rank) and one timing row per (scope, method), appended to `timing` if given (so skip()
    can read the rates measured so far). pool_mask: pool rows that run the reverse search (a shard of the pool;
    default all). Forward and rare searches run for the query_mask rows only, so a shard = (S1 slice, pool slice)
    and the union of all shards equals one full run. reverse_keep: S1 rows whose reverse hits are returned
    (default query_mask; with pool shards pass ALL wanted S1, since any S1 can be hit from any pool shard)."""
    ts, tp = texts(s1), texts(pool)
    qrows = np.flatnonzero(query_mask)
    bk = {sc: buckets(s1, pool, sc) for sc in scopes}
    parts, timing = [], ([] if timing is None else timing)
    for rep in FORWARD:
        t0 = time.time()
        v = fit_vectorizer(fit_corpus[rep])
        S, P = v.transform(ts[rep].to_list()), v.transform(tp[rep].to_list())
        t1 = time.time()
        log(f"  {rep}: fit+transform {t1 - t0:.0f}s (vocab {len(v.vocabulary_):,}, pool nnz/row {P.nnz / max(P.shape[0], 1):.1f})")
        timing.append({"scope": "all", "method": f"{rep}_fit_transform", "search_s": t1 - t0, "queries": len(qrows),
                       "pool": P.shape[0], "pairs_evaluated": None})
        if gpu_check and USE_GPU:
            chk = check_gpu(S[qrows], P, k)
            log(f"  GPU check: {chk}")
            if chk["rank_score_diff"] > 1e-4 or chk["pair_score_diff"] > 1e-4:
                raise RuntimeError(f"GPU top-k disagrees with CPU: {chk}")
            use_gpu(chk["gpu_ns_per_pair"] < chk["cpu_ns_per_pair"])  # the faster device does all searches
            log(f"  search device from now on: {'GPU' if USE_GPU else 'CPU (faster than GPU on this data)'}")
            timing.append({"scope": "all", "method": "device_check", **chk, "device": "GPU" if USE_GPU else "CPU"})
            gpu_check = False
        for sc in scopes:
            s_bucket, p_bucket = bk[sc]
            pairs = _pairs_evaluated(s_bucket[qrows], p_bucket)
            if skip(sc, rep, pairs):
                log(f"  [{sc}] {rep}: skipped ({pairs:.2e} pairs over budget)")
                timing.append({"scope": sc, "method": rep, "pairs_evaluated": pairs, "skipped": True})
            else:
                t2 = time.time()
                f = forward(S[qrows], s_bucket[qrows], P, p_bucket, k)
                f = f.with_columns(pl.Series("qi", qrows[f["qi"].to_numpy()]))  # query-local row -> S1 row
                parts.append(f.with_columns(pl.lit(sc).alias("scope"), pl.lit(rep).alias("method")))
                timing.append({"scope": sc, "method": rep, "search_s": time.time() - t2, "queries": len(qrows),
                               "pool": P.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] {rep}: forward {time.time() - t2:.0f}s, {pairs:.2e} pairs, {f.height:,} rows")
            if rep == REVERSE_REP:
                prow = np.arange(P.shape[0]) if pool_mask is None else np.flatnonzero(pool_mask)
                pairs = _pairs_evaluated(p_bucket[prow], s_bucket, with_none=False)  # NONE rows: all S1
                if skip(sc, "reverse", pairs):
                    log(f"  [{sc}] reverse: skipped ({pairs:.2e} pairs over budget)")
                    timing.append({"scope": sc, "method": "reverse", "pairs_evaluated": pairs, "skipped": True})
                    continue
                t3 = time.time()
                r = reverse(S, s_bucket, P[prow], p_bucket[prow], m, keep_s=query_mask if reverse_keep is None else reverse_keep)
                r = r.with_columns(pl.Series("pi", prow[r["pi"].to_numpy()], dtype=pl.Int64))  # shard row -> pool row
                parts.append(r.with_columns(pl.lit(sc).alias("scope"), pl.lit("reverse").alias("method")))  # has "best"
                timing.append({"scope": sc, "method": "reverse", "search_s": time.time() - t3,
                               "queries": len(prow), "pool": S.shape[0], "pairs_evaluated": pairs})
                log(f"  [{sc}] reverse: {time.time() - t3:.0f}s, {pairs:.2e} pairs, {r.height:,} rows")
        del S, P
    t5 = time.time()
    rr = rare_tokens(s1["core_name"][qrows], pool["core_name"], df_counts)
    rr = rr.with_columns(pl.Series("qi", qrows[rr["qi"].to_numpy()], dtype=pl.Int64), pl.lit("rare").alias("method"))
    parts += [rr.with_columns(pl.lit(sc).alias("scope")) for sc in scopes]  # scope-independent
    timing.append({"scope": "all", "method": "rare", "search_s": time.time() - t5, "queries": len(qrows),
                   "pool": pool.height, "pairs_evaluated": None})
    log(f"  rare: {time.time() - t5:.0f}s")
    cols = ["scope", "qi", "pi", "method", "score", "rank", "best"]
    parts = [p if "best" in p.columns else p.with_columns(pl.lit(None, dtype=pl.Float32).alias("best")) for p in parts]
    return pl.concat([p.select(cols) for p in parts]), timing


def _pairs_evaluated(q_bucket: np.ndarray, p_bucket: np.ndarray, with_none: bool = True) -> float:
    """Number of (query, pool) similarity evaluations implied by the bucket sizes (cost model)."""
    pb = pl.Series(p_bucket).value_counts()
    sizes = dict(zip(pb[pb.columns[0]].to_list(), pb["count"].to_list()))
    n_none = sizes.get(NONE, 0) if with_none else 0
    total = 0.0
    for b, n in zip(*np.unique(q_bucket, return_counts=True)):
        total += n * (len(p_bucket) if b == NONE else sizes.get(b, 0) + n_none)
    return total


In [ ]:
%%writefile /tmp/src/candidates.py
"""Candidate selection on top of blocking: turn the union of search results into a SMALL list per S1.

The candidate set size is part of the ranking (organiser update), so a list should be close to the S1's true
match count (~3.4 on average). Three stages, all tuned on validation:
1. Selection rules on the blocking table (one row per S1 x candidate, rank_*/score_* per method):
   - reverse: each S2/S3 record keeps its top-1 S1, and its top-2..m S1 only if within `rev_margin` of its best;
   - forward: rank <= fwd_k and score within `fwd_margin` of the S1's best for that method (or >= `fwd_min`);
   - rare-token hits (rank <= rare_k);
   - a cap on the list length per S1, ordered by the best cosine over methods.
2. Optional pruner: a small gradient-boosted model on cheap features (blocking scores, ranks, margins,
   agreement of methods, rapidfuzz on core name and address, same city/state). It is trained on candidates of
   TRAIN S1 only and keeps a candidate if p >= tau, up to a cap. Its output is the submitted candidate set.
3. List statistics: recall, candidates per S1 (average, median, p95, S1 with no candidates included) and
   reduction ratio against all S1 x pool pairs of the country.
"""
from __future__ import annotations

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
FORWARD = ("name", "name_city", "name_addr")
METHODS = (*FORWARD, "reverse", "rare")
COSINE = (*FORWARD, "reverse")
RANK_NA = 10**6


def add_best(wide: pl.DataFrame) -> pl.DataFrame:
    """best_<method>: the S1's top forward score per method (margin reference). Needs rank/score columns."""
    return wide.with_columns(*[pl.col(f"score_{m}").max().over("scope", "s1").alias(f"best_{m}") for m in FORWARD],
                             pl.max_horizontal(*[pl.col(f"score_{m}") for m in COSINE]).fill_null(0.0).alias("order"))


def rule(rev_m: int = 0, rev_margin: float | None = None, fwd_k: int = 0, fwd_margin: float | None = None,
         fwd_min: float | None = None, rare_k: int = 0) -> pl.Expr:
    """Boolean expression over the blocking table (after add_best) for one selection config."""
    e = pl.lit(False)
    if rev_m:
        near = pl.lit(True) if rev_margin is None else pl.col("score_reverse") >= pl.col("best_reverse") - rev_margin
        e = e | ((pl.col("rank_reverse") == 1) | ((pl.col("rank_reverse") <= rev_m) & near)).fill_null(False)
    if fwd_k:
        for m in FORWARD:
            ok = pl.col(f"rank_{m}") <= fwd_k
            if fwd_margin is not None:
                near = pl.col(f"score_{m}") >= pl.col(f"best_{m}") - fwd_margin
                if fwd_min is not None:
                    near = near | (pl.col(f"score_{m}") >= fwd_min)
                ok = ok & near
            e = e | ok.fill_null(False)
    if rare_k:
        e = e | (pl.col("rank_rare") <= rare_k).fill_null(False)
    return e


def select(wide: pl.DataFrame, expr: pl.Expr, cap: int | None = None, order: str = "order") -> pl.DataFrame:
    """Rows passing expr, then at most `cap` per (scope, S1) by descending `order`."""
    out = wide.filter(expr)
    if cap:
        out = out.filter(pl.col(order).rank("ordinal", descending=True).over("scope", "s1") <= cap)
    return out


# ---------------------------------------------------------------- pruner

def features(cand: pl.DataFrame, s1_attr: pl.DataFrame, pool_attr: pl.DataFrame) -> pl.DataFrame:
    """Cheap per-pair features. cand: blocking rows (after add_best); *_attr: id + core_name, addr_norm, city, state."""
    a = s1_attr.select(pl.col("entity_id").alias("s1"), *[pl.col(c).alias(f"q_{c}") for c in ("core_name", "addr_norm", "city", "state")])
    b = pool_attr.select(pl.col("entity_id").alias("cand"), *[pl.col(c).alias(f"p_{c}") for c in ("core_name", "addr_norm", "city", "state")])
    x = cand.join(a, on="s1", how="left").join(b, on="cand", how="left")
    fz = lambda scorer, c: process.cpdist(x[f"q_{c}"].fill_null("").to_list(), x[f"p_{c}"].fill_null("").to_list(),
                                          scorer=scorer, workers=-1).astype(np.float32)
    x = x.with_columns(
        pl.Series("f_name_tsr", fz(fuzz.token_set_ratio, "core_name")),
        pl.Series("f_name_ratio", fz(fuzz.ratio, "core_name")),
        pl.Series("f_addr_tsr", fz(fuzz.token_set_ratio, "addr_norm")),
        (pl.col("q_city") == pl.col("p_city")).cast(pl.Int8).fill_null(-1).alias("f_same_city"),
        (pl.col("q_state") == pl.col("p_state")).cast(pl.Int8).fill_null(-1).alias("f_same_state"),
    )
    return x


def feature_cols() -> list[str]:
    return [*[f"score_{m}" for m in METHODS], *[f"rank_{m}" for m in METHODS], *[f"gap_{m}" for m in FORWARD],
            "gap_reverse", "n_methods", "list_rank", "list_size",
            "f_name_tsr", "f_name_ratio", "f_addr_tsr", "f_same_city", "f_same_state"]


def matrix(x: pl.DataFrame) -> np.ndarray:
    """Feature matrix; x must carry list_rank / list_size (add_list_features on the whole list)."""
    x = x.with_columns(
        *[(pl.col(f"best_{m}") - pl.col(f"score_{m}")).alias(f"gap_{m}") for m in FORWARD],
        (pl.col("best_reverse") - pl.col("score_reverse")).alias("gap_reverse"),
        pl.sum_horizontal(*[pl.col(f"rank_{m}").is_not_null() for m in METHODS]).alias("n_methods"),
    ).with_columns(*[pl.col(f"rank_{m}").fill_null(RANK_NA) for m in METHODS])
    return x.select(feature_cols()).to_numpy().astype(np.float32)  # NaN (missing score) is handled by the model


def train_pruner(x: pl.DataFrame, y: np.ndarray) -> HistGradientBoostingClassifier:
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=50,
                                       l2_regularization=1.0, random_state=SEED)
    return m.fit(matrix(x), y)


def add_list_features(cand: pl.DataFrame) -> pl.DataFrame:
    """Per-S1 list features, computed on the WHOLE starting list (before any sampling or chunking)."""
    return cand.with_columns(pl.col("order").rank("ordinal", descending=True).over("scope", "s1").alias("list_rank"),
                             pl.len().over("scope", "s1").alias("list_size"))


def score(model, cand: pl.DataFrame, s1_attr: pl.DataFrame, pool_attr: pl.DataFrame, chunk: int = 2_000_000) -> np.ndarray:
    """Pruner probability per row of cand (after add_list_features), in row chunks to bound memory."""
    return np.concatenate([model.predict_proba(matrix(features(cand[lo:lo + chunk], s1_attr, pool_attr)))[:, 1]
                           for lo in range(0, cand.height, chunk)] or [np.empty(0)])


def training_rows(cand: pl.DataFrame, max_rows: int, seed: int = SEED) -> pl.DataFrame:
    """All positives plus a seeded sample of negatives, at most max_rows (after add_list_features)."""
    if cand.height <= max_rows:
        return cand
    pos, neg = cand.filter(pl.col("is_match")), cand.filter(~pl.col("is_match"))
    return pl.concat([pos, neg.sample(max(max_rows - pos.height, 0), seed=seed)])


def prune(x: pl.DataFrame, p: np.ndarray, tau: float, cap: int | None = None) -> pl.DataFrame:
    """Keep candidates with pruner probability >= tau, at most `cap` per (scope, S1) by descending probability."""
    out = x.with_columns(pl.Series("p_keep", p)).filter(pl.col("p_keep") >= tau)
    if cap:
        out = out.filter(pl.col("p_keep").rank("ordinal", descending=True).over("scope", "s1") <= cap)
    return out


# ---------------------------------------------------------------- list statistics

def list_stats(found: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, pool_size: dict[str, int]) -> pl.DataFrame:
    """found: (s1, cand) selected; truth: (s1, cand) true pairs; s1s: (s1, country) of all evaluated S1;
    pool_size: S2+S3 records per country. Per country + ALL: recall, avg/median/p95 candidates per S1 (S1 with
    no candidates count as 0), share of S1 with an empty list, reduction ratio vs all S1 x pool pairs."""
    per_s1 = lambda df, name: df.join(s1s.select("s1"), on="s1", how="semi").group_by("s1").len(name)
    d = (s1s.join(per_s1(found, "n"), on="s1", how="left")
         .join(per_s1(truth, "true"), on="s1", how="left")
         .join(per_s1(truth.join(found.select("s1", "cand"), on=["s1", "cand"], how="semi"), "hit"), on="s1", how="left")
         .with_columns(pl.col("n", "true", "hit").fill_null(0),
                       pl.col("country").replace_strict(pool_size, default=0, return_dtype=pl.Float64).alias("pool")))
    d = pl.concat([d, d.with_columns(pl.lit("ALL").alias("country"))])
    return d.group_by("country").agg(
        (pl.col("hit").sum() / pl.col("true").sum()).alias("recall"),
        pl.col("n").mean().alias("avg"), pl.col("n").median().alias("median"),
        pl.col("n").quantile(0.95, "nearest").alias("p95"), (pl.col("n") == 0).mean().alias("empty_share"),
        (1 - pl.col("n").sum() / pl.col("pool").sum()).alias("reduction_ratio"),
        pl.len().alias("n_s1"), pl.col("true").sum(), pl.col("n").sum().alias("cands"),
    ).sort(pl.col("country") == "ALL", "country")


def pareto(points: pl.DataFrame, x: str = "avg", y: str = "recall") -> pl.DataFrame:
    """Configs not dominated on (fewer candidates, higher recall)."""
    p = points.sort(x, -pl.col(y))
    best, keep = -1.0, []
    for v in p[y].to_list():
        keep.append(v > best)
        best = max(best, v)
    return p.filter(pl.Series(keep))


In [ ]:
%%writefile /tmp/src/selection.py
"""Match selection: from scored candidate pairs to the predicted matches per S1, tuned for macro F0.5.

Rules (all tuned on validation, never on test):
- threshold: keep a pair only if its match probability p >= tau;
- one owner: every S2/S3 record matches at most one S1 (EDA: 0 of 7.6M train records match two), so each
  candidate is kept only for the S1 that gives it the highest p;
- margin: keep a pair only if p >= margin * (best p of that S1), which drops weak extra matches next to a strong one.
F0.5 weighs precision twice, so a wrong match costs more than a missed one: the tuned threshold is usually high.

f05_table() is a vectorised version of metric.py (same definition, checked against it in the tests and in
run_model.py): macro F0.5 over ALL evaluated S1, an S1 without predictions scores 1.0 if it has no true match.
"""
from __future__ import annotations

import itertools

import polars as pl

TAUS = [round(0.05 * i, 2) for i in range(2, 20)]  # 0.10 .. 0.95
MARGINS = [0.0, 0.3, 0.5, 0.7]
BETA2 = 0.25


def select(scored: pl.DataFrame, tau: float, margin: float = 0.0, one_owner: bool = True, tau2: float | None = None,
           sim: float = 0.9, owner_delta: float = 0.0) -> pl.DataFrame:
    """scored: s1, cand, p (+ s_sim_name, s_num_eq for the secondary rule). Returns the kept (s1, cand, p) rows.
    tau2: also keep a pair with tau2 <= p < tau if it strongly resembles a confident candidate of the same S1
    (s_sim_name >= sim and the same first house number). owner_delta: with one_owner, an S2/S3 claimed by two S1
    goes to the higher p only if it leads by more than owner_delta; otherwise it is dropped from both."""
    keep = pl.col("p") >= tau
    if tau2 is not None and tau2 < tau:
        keep = keep | ((pl.col("p") >= tau2) & (pl.col("s_sim_name") >= sim) & (pl.col("s_num_eq") >= 1)).fill_null(False)
    x = scored.filter(keep)
    if one_owner:  # ties: the smaller s1 id wins (deterministic)
        x = x.sort(["cand", "p", "s1"], descending=[False, True, False])
        if owner_delta > 0:
            lead = pl.col("p") - pl.col("p").shift(-1).over("cand")
            x = x.with_columns(pl.int_range(pl.len()).over("cand").alias("_r"), lead.fill_null(1.0).alias("_lead"))
            x = x.filter((pl.col("_r") == 0) & (pl.col("_lead") > owner_delta)).drop("_r", "_lead")
        else:
            x = x.unique("cand", keep="first", maintain_order=True)
    if margin > 0:
        x = x.filter(pl.col("p") >= margin * pl.col("p").max().over("s1"))
    return x.select("s1", "cand", "p")


def f05_table(pred: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame) -> pl.DataFrame:
    """Per evaluated S1: n_pred, n_true, tp, precision, recall, f05. pred / truth: s1, cand; s1s: s1 (+ any
    attribute columns, e.g. country), the S1 to evaluate. S1 missing from pred count as empty predictions."""
    keys = s1s.select("s1")
    pred = pred.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    truth = truth.join(keys, on="s1", how="semi").select("s1", "cand").unique()
    tp = pred.join(truth, on=["s1", "cand"], how="semi").group_by("s1").len("tp")
    x = (s1s.join(pred.group_by("s1").len("n_pred"), on="s1", how="left")
         .join(truth.group_by("s1").len("n_true"), on="s1", how="left")
         .join(tp, on="s1", how="left")
         .with_columns(pl.col("n_pred", "n_true", "tp").fill_null(0)))
    prec = pl.when(pl.col("n_pred") > 0).then(pl.col("tp") / pl.col("n_pred"))
    rec = pl.when(pl.col("n_true") > 0).then(pl.col("tp") / pl.col("n_true"))
    f = (pl.when(pl.col("n_true") == 0).then((pl.col("n_pred") == 0).cast(pl.Float64))
         .when(pl.col("tp") == 0).then(0.0)
         .otherwise((1 + BETA2) * prec * rec / (BETA2 * prec + rec)))
    return x.with_columns(prec.alias("precision"), rec.alias("recall"), f.alias("f05"))


def summary(tab: pl.DataFrame, by: str | None = None) -> pl.DataFrame:
    """Macro F0.5, mean precision / recall (over S1 where defined), per group and overall (+ singletons)."""
    aggs = [pl.len().alias("s1"), pl.col("f05").mean().alias("macro_f05"), pl.col("precision").mean().alias("precision"),
            pl.col("recall").mean().alias("recall"), pl.col("n_pred").mean().alias("pred_per_s1")]
    kind = pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton"))
    parts = [tab.select(pl.lit("overall").alias("group"), *aggs),
             tab.with_columns(kind.alias("_k")).group_by("_k").agg(aggs).sort("_k")
             .select(pl.format("kind={}", "_k").alias("group"), *[a.meta.output_name() for a in aggs])]
    if by:
        parts.insert(1, tab.group_by(by).agg(aggs).sort(by).select(pl.format(f"{by}={{}}", by).alias("group"),
                                                                     *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


def tune(scored: pl.DataFrame, truth: pl.DataFrame, s1s: pl.DataFrame, taus=TAUS, margins=MARGINS,
         owners=(True, False)) -> pl.DataFrame:
    """Macro F0.5 on s1s for every (tau, margin, one_owner). Sorted best first."""
    rows = []
    for tau, margin, owner in itertools.product(taus, margins, owners):
        tab = f05_table(select(scored, tau, margin, owner), truth, s1s)
        rows.append({"tau": tau, "margin": margin, "one_owner": owner, "macro_f05": tab["f05"].mean(),
                     "pred_per_s1": tab["n_pred"].mean()})
    return pl.DataFrame(rows).sort("macro_f05", descending=True)


In [ ]:
%%writefile /tmp/src/submission.py
"""Write matching_results.tsv / candidate_pairs.tsv in the exact challenge format.

One row per S1 (in the given order), tab-separated, comma-joined IDs, no quoting,
no duplicates, empty second column when there is nothing to list.
"""
from collections.abc import Iterable, Mapping
from pathlib import Path

MATCHING_HEADER = ("source1_entity_id", "matched_entity_ids")
CANDIDATE_HEADER = ("source1_entity_id", "candidate_entity_ids")


def write_id_lists(path: Path, header: tuple[str, str], s1_ids: Iterable[str],
                   lists: Mapping[str, Iterable[str]]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    n = 0
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s1 in s1_ids:
            ids = list(dict.fromkeys(i for i in lists.get(s1, ()) if not i.startswith("S1-")))
            f.write(f"{s1}\t{','.join(ids)}\n")
            n += 1
    return n


def write_submission(out_dir: Path, s1_ids: list[str], matches: Mapping[str, Iterable[str]],
                     candidates: Mapping[str, Iterable[str]]) -> None:
    for s1, m in matches.items():
        extra = set(m) - set(candidates.get(s1, ()))
        if extra:
            raise ValueError(f"{s1}: matches not in candidates: {sorted(extra)[:5]}")
    write_id_lists(out_dir / "matching_results.tsv", MATCHING_HEADER, s1_ids, matches)
    write_id_lists(out_dir / "candidate_pairs.tsv", CANDIDATE_HEADER, s1_ids, candidates)


In [ ]:
%%writefile /tmp/src/metric.py
"""Macro F0.5 exactly as the challenge defines it.

Per S1 entity: F0.5 = 1.25*P*R / (0.25*P + R).
- No true matches (singleton): 1.0 if the prediction is empty, else 0.0.
- True matches but empty prediction, or no correct IDs: 0.0.
Macro average over ALL S1 entities in the evaluation set; an S1 missing from the
predictions counts as an empty prediction.
"""
from collections.abc import Iterable, Mapping

import polars as pl

BETA2 = 0.25  # beta = 0.5


def parse_ids(s: str | None) -> set[str]:
    """'S2-1,S3-2' -> {'S2-1', 'S3-2'}; '' or None -> set()."""
    return {x.strip() for x in s.split(",") if x.strip()} if s else set()


def f05(pred: Iterable[str], truth: Iterable[str]) -> float:
    pred, truth = set(pred), set(truth)
    if not truth:
        return 1.0 if not pred else 0.0
    tp = len(pred & truth)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(truth)
    return (1 + BETA2) * p * r / (BETA2 * p + r)


def per_entity(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]]) -> pl.DataFrame:
    """One row per S1 in `truth`: f05, precision, recall (P/R null when undefined)."""
    rows = []
    for s1, t in truth.items():
        p, t = set(pred.get(s1, ())), set(t)
        tp = len(p & t)
        rows.append({
            "s1": s1,
            "f05": f05(p, t),
            "precision": tp / len(p) if p else None,
            "recall": tp / len(t) if t else None,
            "n_true": len(t),
            "n_pred": len(p),
        })
    return pl.DataFrame(rows, schema={"s1": pl.String, "f05": pl.Float64, "precision": pl.Float64,
                                      "recall": pl.Float64, "n_true": pl.Int64, "n_pred": pl.Int64})


def report(pred: Mapping[str, Iterable[str]], truth: Mapping[str, Iterable[str]],
           country: Mapping[str, str] | None = None) -> pl.DataFrame:
    """Macro F0.5 overall, per country, and for singletons vs non-singletons."""
    df = per_entity(pred, truth).with_columns(
        pl.when(pl.col("n_true") == 0).then(pl.lit("singleton")).otherwise(pl.lit("non_singleton")).alias("kind"),
        pl.col("s1").replace_strict(country or {}, default="ALL").alias("country"),
    )
    aggs = [
        pl.len().alias("s1"),
        pl.col("f05").mean().alias("macro_f05"),
        pl.col("precision").mean().alias("mean_precision"),
        pl.col("recall").mean().alias("mean_recall"),
    ]
    parts = [df.select(pl.lit("overall").alias("group"), *aggs)]
    if country:
        parts.append(df.group_by("country").agg(aggs).sort("country").select(pl.format("country={}", "country").alias("group"), *[a.meta.output_name() for a in aggs]))
    parts.append(df.group_by("kind").agg(aggs).sort("kind").select(pl.format("kind={}", "kind").alias("group"), *[a.meta.output_name() for a in aggs]))
    return pl.concat(parts, how="vertical_relaxed")


In [ ]:
%%writefile /tmp/src/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator

Run this BEFORE submitting. It checks your output files against every formatting
rule the scorer enforces, so you can catch a rejection locally instead of burning
a submission. It reads only your output files and the test source files (to learn
which S1 entities are required and which S2/S3 IDs exist); it never needs the
ground truth and never computes your score.

It validates two files:

* ``matching_results.tsv`` (required) — your final matches, the file scored on the
  leaderboard.
* ``candidate_pairs.tsv`` (optional) — the candidate set from your blocking stage.
  When present, the validator also checks that your final matches are a subset of
  your candidates and *warns* (never fails) otherwise. When absent it is skipped
  with a warning; it is still expected in your final submission zip.

Stdlib only, Python 3.8+. Run from the ``student_resource/`` directory::

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 means the files are safe to submit; 1 means fix the listed issues
(warnings never fail the run).

ID-existence check (off by default). By default the validator does NOT check that
every matched/candidate ID actually exists in the test set: that check loads all
Source-2/3 IDs into memory, which on the full ~1.7M-entity test set costs a few GB
(more when ``candidate_pairs.tsv`` is included). The default run therefore stays fast
and light and verifies every other rule; it prints a warning noting the check was
skipped. Pass ``--check-ids`` to turn it on (it reads ``test_source2.tsv`` /
``test_source3.tsv`` from ``--test-dir``); a missing/garbage matched ID only lowers
your score rather than being rejected by the scorer, so this check is a diagnostic,
not a gate. If ``--check-ids`` runs out of memory, drop ``--candidate`` (the candidate
cross-check is the biggest memory user, and the matching file is the only one scored).
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5  # how many offending IDs to show per issue
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    """Return the set of first-column entity IDs from a source TSV.

    The header row is skipped and blank lines are ignored.
    """
    with open(path, encoding="utf-8") as f:
        next(f, None)  # skip header
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    """Return a short, human-readable sample of ``items`` for an error message."""
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    """Return the set of valid S2/S3 match IDs, or ``None`` if unavailable.

    Only called when ``--check-ids`` is on. When ``test_source2.tsv`` or
    ``test_source3.tsv`` is missing we cannot check that matched IDs exist, so we
    record a warning and return ``None`` to signal that the existence check should be
    skipped.
    """
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set. Every other rule is still checked. "
                f"This is the lighter-memory mode; provide test_source2/3.tsv to "
                f"enable the ID-existence check."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    """Validate one results-style TSV (matching or candidate).

    Applies the shared formatting rules and appends any problems to ``errors``.
    Returns a ``{source1_id: set(matched/candidate ids)}`` mapping, or ``None`` on a
    fatal problem (missing file, empty file, or a broken header) that stops parsing.
    """
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:  # the #1 mistake: a CSV
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv); "
                "write it with df.to_csv(sep='\\t', index=False)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    # Aggregate the per-category findings. Each entry is (offenders, message);
    # only non-empty categories become errors.
    findings = [
        (
            dup_rows,
            "{name}: duplicate source1_entity_id row(s): {ex}. "
            "Each S1 entity may appear on only one row.",
        ),
        (
            intra_dupes,
            "{name}: repeated ID inside a {col} list for: {ex}. "
            "No duplicate IDs are allowed within a list.",
        ),
        (
            self_matches,
            "{name}: {col} contains Source-1 IDs (self-matches): {ex}. "
            "Only S2-/S3- IDs are allowed.",
        ),
        (
            wrong_prefix,
            "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}.",
        ),
        (
            unknown,
            "{name}: {col} references IDs not in the test "
            "Source-2/3 files: {ex}.",
        ),
        (
            required - seen,
            "{name}: required S1 entity(ies) missing: {ex}. "
            "Every entity in test_source1.tsv needs a row (empty = no match).",
        ),
        (
            seen - required,
            "{name}: row(s) using an S1 ID that is not in the test set: {ex}.",
        ),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    """Validate the submission output(s); return ``(errors, warnings)`` lists.

    ``check_ids`` (``--check-ids``) turns on the optional, memory-heavy check that
    every matched/candidate ID exists in the test Source-2/3 files. It is off by
    default so the common run stays fast and light.
    """
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default) — not checking that matched/"
            "candidate IDs exist in the test set. Every other rule is still checked. "
            "Re-run with --check-ids to enable it (needs test_source2/3.tsv; uses "
            "more memory). A nonexistent ID only lowers your score, never rejects "
            "your submission."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    # candidate_pairs.tsv is optional: if it's absent we skip its checks with a
    # warning (it's still expected in your final submission zip). A missing
    # candidate file never fails this run on its own.
    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks. "
            "It is optional here, but your final submission zip must include "
            "output/candidate_pairs.tsv."
        )

    # Soft check: your final matches should come from your blocking candidates.
    # A matched ID absent from candidate_pairs.tsv usually means a pipeline bug,
    # so we warn but never fail on it.
    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}. Final matches "
                "normally come from your blocking candidates — double-check these."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate ML Challenge 2026 submission output files before submitting."
    )
    parser.add_argument(
        "--matching",
        "-m",
        default="output/matching_results.tsv",
        help="Path to matching_results.tsv (default: %(default)s)",
    )
    parser.add_argument(
        "--candidate",
        "-c",
        default=None,
        help="Path to candidate_pairs.tsv "
        "(default: output/candidate_pairs.tsv if it exists).",
    )
    parser.add_argument(
        "--test-dir",
        "-t",
        default="dataset/test",
        help="Folder with test_source1/2/3.tsv (default: %(default)s). "
        "test_source2/3.tsv are only read when --check-ids is given.",
    )
    parser.add_argument(
        "--check-ids",
        action="store_true",
        help="Also check that every matched/candidate ID exists in the test "
        "Source-2/3 files. Off by default (loads all S2/S3 IDs into memory — a few "
        "GB on the full test set). A nonexistent ID only lowers your score, so this "
        "is a diagnostic, not a submission gate.",
    )
    args = parser.parse_args()

    # candidate_pairs.tsv is optional; default to the conventional path and let
    # validate() skip (with a warning) if the file isn't there.
    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(
            f"  1. A file is not valid UTF-8 text (most likely {args.matching} or "
            f"{candidate_path}). Re-save it as a plain UTF-8, tab-separated .tsv — "
            "not cp1252/Latin-1, and not a compressed or binary file (.gz/.xlsx/"
            ".parquet) renamed to .tsv. In pandas: "
            "df.to_csv(path, sep='\\t', index=False, encoding='utf-8')."
        )
        return 1
    except OSError as exc:
        print()
        print("FAIL — 1 issue(s) to fix before submitting:")
        print(f"  1. Could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_final.py', '--input', '/kaggle/input', '--config', '/kaggle/input', '--work', '/kaggle/working', '--validator', '/tmp/src/validate_submission.py']

In [ ]:
"""Final model end to end: train_model.py (train + validation tune half, report-half check) then predict.py
(test outputs, validator, French checks). One entry point for the Kaggle job and for reproduction.

  python src/run_final.py --input DIR --config best_config.json --work DIR [--validator PATH] [--fallback PATH]
"""
import argparse
import os
import subprocess
import sys
from pathlib import Path

try:
    SRC = Path(__file__).resolve().parent
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    SRC = Path("/tmp/src")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--work", required=True)
    ap.add_argument("--validator", default=None)
    ap.add_argument("--fallback", default=None)
    a = ap.parse_args()
    env = {**os.environ, "PYTHONPATH": str(SRC)}
    model_dir, out_dir = Path(a.work) / "model_final", Path(a.work) / "output"
    run = lambda args: subprocess.run([sys.executable, *args], check=True, cwd=str(SRC), env=env)
    run([str(SRC / "train_model.py"), "--input", a.input, "--config", a.config, "--out-dir", str(model_dir), "--final"])
    run([str(SRC / "predict.py"), "--input", a.input, "--model-dir", str(model_dir), "--out-dir", str(out_dir)]
        + (["--validator", a.validator] if a.validator else []) + (["--fallback", a.fallback] if a.fallback else []))


if __name__ == "__main__":
    main()
